# Hito 2 · Modelado inicial

Comparación inicial con conversión verificada de UTC a hora de Lima.

## 1. Business Understanding


### Entorno y configuración

El CSV de SENAMHI registra las horas en UTC. Se convierten a America/Lima al unir el aire con las demás fuentes y al construir las variables de calendario.

In [ ]:
import importlib.util, subprocess, sys
requeridos = {'jinja2':'jinja2>=3,<4', 'polars':'polars>=1.32,<2', 'pyarrow':'pyarrow', 'statsmodels':'statsmodels>=0.14,<0.16',
              'plotly':'plotly>=5,<7', 'xgboost':'xgboost>=2,<4', 'lightgbm':'lightgbm>=4,<5',
              'catboost':'catboost>=1.2,<2'}
faltan=[spec for modulo,spec in requeridos.items() if importlib.util.find_spec(modulo) is None]
if faltan: subprocess.check_call([sys.executable,'-m','pip','install','-q',*faltan])


In [ ]:
from pathlib import Path
import csv
import math
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid")

import zipfile, json, time, warnings, sklearn, statsmodels
from IPython.display import Markdown, HTML
from threadpoolctl import threadpool_limits
get_ipython().run_line_magic('matplotlib', 'inline')

# Cambiar PM25_DATA_DIR
BASE = Path(os.environ.get('PM25_DATA_DIR', '/content' if Path('/content').exists() else Path.cwd()))
HORA_SENAMHI = 'UTC_VERIFICADA'
SEED = 42
N_JOBS = 4
np.random.seed(SEED)
threadpool_limits(limits=N_JOBS)
SALIDA = BASE / 'hito2_resultados'
SALIDA.mkdir(parents=True, exist_ok=True)
for z in [BASE/'csv_complementarios_pm25_lima.zip', BASE/'upload'/'csv_complementarios_pm25_lima.zip']:
    if z.exists() and not (BASE/'datos_complementarios_pm25'/'datos_complementarios_pm25_2015_2024.csv').exists():
        with zipfile.ZipFile(z) as paquete:
            if any(Path(n).name != n for n in paquete.namelist()): raise ValueError('ZIP con rutas no previstas')
            paquete.extractall(BASE/'datos_complementarios_pm25')

def localizar(*nombres):
    carpetas=[BASE,BASE/'upload',BASE/'datos',BASE/'datos_complementarios_pm25']
    for nombre in nombres:
        for carpeta in carpetas:
            p=carpeta/nombre
            if p.is_file(): return p
    raise FileNotFoundError('Falta uno de estos archivos: '+str(nombres)+'; revisar BASE.')

RUTA_AIRE=localizar('aire_lima_completo.csv','aire_lima_completo(1).csv')
RUTA_CLIMA=localizar('clima_era5_7_estaciones.parquet','clima_era5_7_estaciones(1).parquet')
RUTA_DEMO_CENSO=localizar('poblacion_censo2017_7_distritos.csv')
RUTA_DEMO_PROY=localizar('poblacion_proyectada_2018_2026_7_distritos.csv')
RUTA_COMPLEMENTOS=localizar('datos_complementarios_pm25_2015_2024.csv')
RUTA_CALOR=localizar('focos_calor_lima_entorno_2015_2024.csv')
RUTA_VEHICULOS=localizar('tenencia_vehicular_7_distritos_censo2017.csv')
ARCHIVOS_PROYECTO=(RUTA_AIRE,RUTA_CLIMA,RUTA_DEMO_CENSO,RUTA_DEMO_PROY,
                  RUTA_COMPLEMENTOS,RUTA_CALOR,RUTA_VEHICULOS)
for ruta in ARCHIVOS_PROYECTO: print(ruta.name, round(ruta.stat().st_size/1e6,2),'MB')
pd.set_option('display.max_columns',25)
pd.set_option('display.precision',4)
print('Versiones:', {'pandas':pd.__version__,'sklearn':sklearn.__version__,
                      'statsmodels':statsmodels.__version__,'polars':pl.__version__})
print('Referencia horaria del aire:', HORA_SENAMHI)

CONTAMINANTES = ["PM2_5", "PM10", "NO2"]
METEOROLOGIA = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "pressure_msl", "wind_speed_10m", "wind_direction_10m",
    "boundary_layer_height",
]
NUMERICAS_AIRE = CONTAMINANTES + ["LATITUD", "LONGITUD", "ALTITUD"]

def mostrar(tabla, titulo, limite=20):
    print("\n" + titulo)
    if isinstance(tabla, pd.DataFrame):
        display(tabla.head(limite))
    else:
        display(tabla.head(limite).to_pandas())

def figura(nombre):
    plt.tight_layout()
    plt.savefig(SALIDA / nombre, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()




## 2. Data Understanding & Exploratory Data Analysis

### 2.1. Fuentes originales y controles de integridad
Conservamos el alcance del EDA del Hito 1 y se recalcula con hora Perú. Los controles de tipado permiten estudiar las fuentes sin transformar las ausencias en mediciones. Los resúmenes base usan sus identificadores originales; las nuevas tablas continúan desde la Tabla 12 y las figuras desde la Figura 12 para así nosotros mantener un orden con el informe final.

### Fuentes del hito 1 que estan separados


### Calidad del aire: CSV completo


### Inventario, estructura y muestra del CSV de aire


In [ ]:
with RUTA_AIRE.open(encoding="utf-8-sig", newline="") as archivo:
    encabezado = next(csv.reader(archivo))
print("Columnas del CSV:", len(encabezado), encabezado)
assert len(encabezado) == len(set(encabezado)), "El CSV tiene nombres de columna duplicados."
obligatorias = {"ESTACION", "FECHA", "HORA", "PM2_5", "PM10", "NO2"}
assert obligatorias.issubset(encabezado), f"Faltan: {obligatorias - set(encabezado)}"

raw = pl.scan_csv(
    str(RUTA_AIRE), infer_schema=False, null_values=["NA", "N/A", "null"],
    ignore_errors=False, truncate_ragged_lines=False
)
esquema = pl.DataFrame({
    "columna": encabezado,
    "tipo_lectura": [str(raw.collect_schema()[c]) for c in encabezado],
    "tipo_esperado": [
        "flotante" if c in NUMERICAS_AIRE else
        "fecha AAAAMMDD" if c in {"FECHA", "FECHA_CORTE"} else
        "hora HHMMSS" if c == "HORA" else "texto/identificador"
        for c in encabezado
    ],
})
mostrar(esquema, "Resumen base 1 — Diccionario técnico: tipo leído y tipo esperado")
mostrar(raw.head(8).collect(), "Resumen base 2 — Primeras filas originales", 8)




### Completitud y cardinalidad de todas las columnas del aire

Se contabilizan filas, no nulos, faltantes y valores distintos **para cada una de las 15 columnas**. Identificadores, ubicación y fecha de corte se revisan junto con PM10, PM2.5 y NO₂.


In [ ]:
conteos = raw.select(
    pl.len().alias("_n"),
    *[pl.col(c).null_count().alias(c) for c in encabezado]
).collect(engine="streaming").row(0, named=True)
n_aire_original = int(conteos.pop("_n"))
completitud = pl.DataFrame([
    {"columna": c, "no_nulos": n_aire_original - int(conteos[c]),
     "nulos": int(conteos[c]),
     "nulos_pct": round(100 * int(conteos[c]) / n_aire_original, 2)}
    for c in encabezado
]).sort("nulos_pct", descending=True)
print("Registros:", n_aire_original, "Columnas:", len(encabezado))
mostrar(completitud, "Resumen base 3 — Datos no nulos y faltantes por columna", 20)

distintos_aire = raw.select([
    pl.col(c).drop_nulls().n_unique().alias(c) for c in encabezado
]).collect(engine="in-memory").row(0, named=True)
cardinalidad_aire = pl.DataFrame([
    {"columna": c, "distintos_no_nulos": int(distintos_aire[c])}
    for c in encabezado
])
mostrar(cardinalidad_aire, "Resumen base 3A — Valores distintos en todas las columnas")
mostrar(
    raw.group_by("ESTACION").len().sort("len", descending=True).collect(engine="streaming"),
    "Resumen base 4 — Número de registros por estación"
)




### Tipos, resumen numérico e integridad del aire

Se calculan mínimos, máximos, media, mediana y cuantiles. Se inspeccionan fechas, horas, duplicados, ID, UBIGEO, fecha de corte, coordenadas y consistencia de los atributos por estación.


In [ ]:
fecha_texto = pl.concat_str([
    pl.col("FECHA").str.strip_chars().str.pad_start(8, "0"),
    pl.col("HORA").str.strip_chars().str.pad_start(6, "0"),
])
tipados = raw.with_columns(
    [
        pl.col(c).cast(pl.Float64, strict=False).alias(c + "_NUM")
        for c in NUMERICAS_AIRE
    ] + [
        fecha_texto.str.strptime(pl.Datetime, "%Y%m%d%H%M%S", strict=False)
            .alias("FECHA_HORA_ORIGINAL")
    ]
)

RUTA_AUDITORIA = SALIDA / "aire_tipado_para_auditoria.parquet"
tipados.sink_parquet(str(RUTA_AUDITORIA), compression="zstd")
tipados = pl.scan_parquet(str(RUTA_AUDITORIA))
print("Auditoría repetible desde:", RUTA_AUDITORIA.name)
incidencias = tipados.select(
    pl.col("FECHA_HORA_ORIGINAL").is_null().sum().alias("fecha_hora_no_interpretable"),
    pl.col("ESTACION").is_null().sum().alias("estacion_ausente"),
    *[
        (pl.col(c).is_not_null() & pl.col(c + "_NUM").is_null())
            .sum().alias("conversion_fallida_" + c)
        for c in NUMERICAS_AIRE
    ],
    *[
        (pl.col(c + "_NUM") < 0).sum().alias("concentraciones_negativas_" + c)
        for c in CONTAMINANTES
    ],
).collect(engine="streaming")
mostrar(incidencias.transpose(
    include_header=True, header_name="control", column_names=["casos"]
), "Resumen base 5 — Incongruencias de tipos y valores", 25)

filas_resumen = []
for c in NUMERICAS_AIRE:
    x = pl.col(c + "_NUM")
    r = tipados.select(
        x.count().alias("n"), x.min().alias("min"), x.quantile(.25, "linear").alias("p25"),
        x.median().alias("mediana"), x.mean().alias("media"),
        x.quantile(.75, "linear").alias("p75"),
        x.quantile(.95, "linear").alias("p95"), x.max().alias("max")
    ).collect(engine="streaming").row(0, named=True)
    filas_resumen.append({"variable": c, **r})
resumen_numerico = pl.DataFrame(filas_resumen)
mostrar(resumen_numerico, "Resumen base 6 — Resumen numérico original: mínimos, máximos y cuantiles")

reglas = tipados.select(
    ((pl.col("LATITUD_NUM") < -13) | (pl.col("LATITUD_NUM") > -11))
        .fill_null(False).sum().alias("latitud_fuera_de_rango_lima"),
    ((pl.col("LONGITUD_NUM") < -78.5) | (pl.col("LONGITUD_NUM") > -76))
        .fill_null(False).sum().alias("longitud_fuera_de_rango_lima"),
    pl.col("FECHA_HORA_ORIGINAL").min().alias("primer_registro"),
    pl.col("FECHA_HORA_ORIGINAL").max().alias("ultimo_registro"),
).collect(engine="streaming")
mostrar(reglas, "Resumen base 7 — Fechas y comprobación espacial")

duplicados_hora = (
    tipados.filter(pl.col("FECHA_HORA_ORIGINAL").is_not_null())
    .group_by(["ESTACION", "FECHA_HORA_ORIGINAL"]).len()
    .filter(pl.col("len") > 1)
    .select(pl.len()).collect().item()
)
duplicados_id = (
    tipados.filter(pl.col("ID").is_not_null())
    .group_by("ID").len().filter(pl.col("len") > 1)
    .select(pl.len()).collect().item()
)
print("Claves estación/hora duplicadas:", duplicados_hora)
print("ID duplicados:", duplicados_id)
integridad_identificadores = tipados.select(
    pl.col("ID").cast(pl.Int64, strict=False).is_null().sum()
      .alias("id_no_numerico"),
    (pl.col("UBIGEO").str.len_chars() != 6).fill_null(True).sum()
      .alias("ubigeo_longitud_distinta_de_seis"),
    pl.col("FECHA_CORTE").str.strptime(pl.Date, "%Y%m%d", strict=False)
      .is_null().sum().alias("fecha_corte_no_interpretable"),
    pl.col("HORA").str.pad_start(6, "0").str.strptime(pl.Time, "%H%M%S", strict=False)
      .is_null().sum().alias("hora_no_interpretable"),
).collect(engine="in-memory")
mostrar(integridad_identificadores, "Resumen base 8A — Integridad de ID, UBIGEO y fechas")
metadatos_por_estacion = (
    tipados.group_by("ESTACION")
    .agg(pl.len().alias("filas"),
         *[pl.col(c).n_unique().alias("distintos_" + c)
           for c in ["DISTRITO", "UBIGEO", "LONGITUD", "LATITUD",
                     "ALTITUD", "FECHA_CORTE"]],
         pl.col("DISTRITO").first().alias("distrito_representativo"))
    .sort("ESTACION").collect(engine="in-memory")
)
mostrar(metadatos_por_estacion,
        "Resumen base 8B — Correspondencia de estación, distrito, ubicación y fecha de corte")
mostrar(
    tipados.group_by("DISTRITO").len().sort("len", descending=True).collect(),
    "Resumen base 8 — Distritos presentes"
)




### Clima: archivo horario


### Estructura, completitud y resumen numérico del clima



In [ ]:
meteo_raw = pl.scan_parquet(str(RUTA_CLIMA))
esquema_meteo = meteo_raw.collect_schema()
assert set(["ESTACION", "FECHA_HORA_LIMA", *METEOROLOGIA]).issubset(
    esquema_meteo.names()
), "El Parquet meteorológico no contiene las columnas esperadas."
print("Esquema meteorológico:", esquema_meteo)
meteo_n = meteo_raw.select(pl.len()).collect(engine="streaming").item()
print("Filas meteorológicas:", meteo_n)
columnas_meteo = esquema_meteo.names()
perfil_columnas_meteo = meteo_raw.select(
    *[pl.col(c).null_count().alias("nulos_" + c) for c in columnas_meteo],
    *[pl.col(c).drop_nulls().n_unique().alias("distintos_" + c)
      for c in columnas_meteo]
).collect(engine="in-memory").row(0, named=True)
completitud_clima = pl.DataFrame([
    {"columna": c, "tipo": str(esquema_meteo[c]),
     "no_nulos": meteo_n - int(perfil_columnas_meteo["nulos_" + c]),
     "nulos": int(perfil_columnas_meteo["nulos_" + c]),
     "distintos_no_nulos": int(perfil_columnas_meteo["distintos_" + c])}
    for c in columnas_meteo
])
mostrar(completitud_clima, "Resumen base 9A — Tipos, completitud y cardinalidad del clima", 15)
meteo_por_estacion = (
    meteo_raw.group_by("ESTACION")
    .agg(pl.len().alias("filas"),
         pl.col("FECHA_HORA_LIMA").min().alias("inicio"),
         pl.col("FECHA_HORA_LIMA").max().alias("fin"),
         pl.col("boundary_layer_height").null_count().alias("nulos_capa_limite"))
    .sort("ESTACION").collect(engine="in-memory")
)
mostrar(meteo_por_estacion, "Resumen base 9B — Cobertura climática por estación")
mostrar(
    meteo_raw.select(
        pl.len().alias("filas"),
        pl.col("ESTACION").n_unique().alias("estaciones"),
        pl.col("FECHA_HORA_LIMA").min().alias("inicio"),
        pl.col("FECHA_HORA_LIMA").max().alias("fin"),
        *[pl.col(c).null_count().alias("faltantes_" + c) for c in METEOROLOGIA],
    ).collect(engine="streaming"),
    "Resumen base 9 — Cobertura temporal y nulos del clima"
)
filas_meteo = []
for c in METEOROLOGIA:
    x = pl.col(c).cast(pl.Float64, strict=False)
    r = meteo_raw.select(
        x.count().alias("no_nulos"), x.min().alias("min"),
        x.median().alias("mediana"), x.mean().alias("media"),
        x.quantile(.95, "linear").alias("p95"), x.max().alias("max")
    ).collect(engine="streaming").row(0, named=True)
    filas_meteo.append({"variable": c, **r})
mostrar(pl.DataFrame(filas_meteo), "Resumen base 10 — Resumen numérico meteorológico")

anomalías_meteo = meteo_raw.select(
    ((pl.col("relative_humidity_2m") < 0) |
     (pl.col("relative_humidity_2m") > 100)).fill_null(False).sum().alias("humedad_imposible"),
    (pl.col("precipitation") < 0).fill_null(False).sum().alias("precipitacion_negativa"),
    (pl.col("wind_speed_10m") < 0).fill_null(False).sum().alias("viento_negativo"),
    ((pl.col("wind_direction_10m") < 0) |
     (pl.col("wind_direction_10m") > 360)).fill_null(False).sum().alias("direccion_invalida"),
    (pl.col("pressure_msl") <= 0).fill_null(False).sum().alias("presion_no_positiva"),
    (pl.col("boundary_layer_height") < 0).fill_null(False).sum().alias("capa_negativa"),
).collect(engine="streaming")
mostrar(anomalías_meteo, "Resumen base 11 — Incongruencias físicas del clima")
duplicados_meteo = (
    meteo_raw.group_by(["ESTACION", "FECHA_HORA_LIMA"]).len()
    .filter(pl.col("len") > 1).select(pl.len()).collect().item()
)
print("Claves meteorológicas duplicadas:", duplicados_meteo)




### Datos demográficos




### Estructura, completitud y resumen fuente **demográfica**

In [ ]:
# 1. Carga inicial desde las rutas
df_censo = pd.read_csv(RUTA_DEMO_CENSO)
df_proy = pd.read_csv(RUTA_DEMO_PROY)

# MOSTRAR LAS COLUMNAS EXACTAS PARA VERIFICACIÓN
print("Columnas en Censo 2017:", df_censo.columns.tolist())
print("Columnas en Proyecciones:", df_proy.columns.tolist())

# 2. Detección de columnas y cálculo previo de variables derivadas
col_total_censo = [c for c in df_censo.columns if 'total' in c.lower() or 'pob' in c.lower()][0]
col_ninos = [c for c in df_censo.columns if ('0' in c or '4' in c) and c != col_total_censo][0]
col_adultos = [c for c in df_censo.columns if ('65' in c or 'mas' in c.lower()) and c != col_total_censo][0]

df_censo['PCT_0_4'] = (df_censo[col_ninos] / df_censo[col_total_censo]) * 100
df_censo['PCT_65_MAS'] = (df_censo[col_adultos] / df_censo[col_total_censo]) * 100

# 3. Diccionarios descriptivos completados
data_description_censo = {
    'ESTACION': 'Nombre de la estación de monitoreo SENAMHI.',
    'UBIGEO': 'Código geográfico oficial del distrito (6 dígitos).',
    'DISTRITO': 'Nombre del distrito.',
    'POBLACION_CENSADA_2017': 'Población censada total en 2017.',
    'EDAD_0_4_2017': 'Población infantil de 0 a 4 años.',
    'EDAD_65_MAS_2017': 'Población de adultos mayores (65+ años).',
    'PORC_0_4_Y_65_MAS_2017': 'Porcentaje acumulado de población vulnerable (0–4 y 65+ años).',
    'PCT_0_4': 'Porcentaje de población infantil de 0 a 4 años.',
    'PCT_65_MAS': 'Porcentaje de adultos mayores (65+ años).'
}

data_description_proy = {
    'UBIGEO': 'Código geográfico del distrito.',
    'DISTRITO_INEI': 'Nombre oficial del distrito INEI.',
    'ANIO': 'Año de la proyección (2018–2024).',
    'POBLACION_PROYECTADA': 'Estimación de población total.'
}

# 4. Función para compilar la tabla de metadatos
def create_metadata_df(df, description_dict, name):
    metadata = []
    for col in df.columns:
        metadata.append({
            'Conjunto de Datos': name,
            'Nombre de Columna': col,
            'Tipo de Dato': str(df[col].dtype),
            'Descripción / Propósito': description_dict.get(col, 'Descripción no disponible.')
        })
    return pd.DataFrame(metadata)

# 5. Generación y unión de metadatos
df_censo_metadata = create_metadata_df(df_censo, data_description_censo, 'Censo 2017')
df_proy_metadata = create_metadata_df(df_proy, data_description_proy, 'Proyecciones 2018–2024')

full_metadata = pd.concat([df_censo_metadata, df_proy_metadata]).reset_index(drop=True)

# 6. Salida de la tabla con el título exacto
mostrar(full_metadata, "Variables procesadas en las bases demográficas (INEI)", limite=50)

In [ ]:
# 1. Estandarizar nombres de distritos
for df_temp in [df_censo, df_proy]:
    col_dist = [c for c in df_temp.columns if 'dist' in c.lower() or 'estac' in c.lower()][0]
    df_temp[col_dist] = (
        df_temp[col_dist]
        .astype(str)
        .str.replace('CAMPO_DE_MARTE', 'JESUS MARIA')
        .str.replace('_', ' ')
        .str.title()
        .str.strip()
    )

# 2. Identificar columnas clave en Censo 2017
col_dist_censo = [c for c in df_censo.columns if 'dist' in c.lower() or 'estac' in c.lower()][0]
col_total_censo = [c for c in df_censo.columns if 'total' in c.lower() or 'pob' in c.lower()][0]

col_ninos = [c for c in df_censo.columns if ('0' in c or '4' in c) and c != col_total_censo][0]
col_adultos = [c for c in df_censo.columns if ('65' in c or 'mas' in c.lower()) and c != col_total_censo][0]

# 3. Recalcular porcentajes de grupos vulnerables (Censo 2017)
df_censo['PCT_0_4'] = (df_censo[col_ninos] / df_censo[col_total_censo]) * 100
df_censo['PCT_65_MAS'] = (df_censo[col_adultos] / df_censo[col_total_censo]) * 100

# 4. Construcción y formato de la tabla final del Censo 2017
cols_tabla_censo = ['UBIGEO', col_dist_censo, col_total_censo, col_ninos, 'PCT_0_4', col_adultos, 'PCT_65_MAS']
tabla_2017 = df_censo[cols_tabla_censo].copy()
tabla_2017.columns = [
    'UBIGEO', 'Distrito', 'Población Total',
    'Pob. 0-4 años', '% 0-4 años',
    'Pob. 65+ años', '% 65+ años'
]

# 5. Construcción y formato de la tabla de Proyecciones Poblacionales
col_dist_proy = [c for c in df_proy.columns if 'dist' in c.lower() or 'estac' in c.lower()][0]
col_pob_proy = [c for c in df_proy.columns if 'pob' in c.lower()][0]
col_anio_proy = [c for c in df_proy.columns if 'an' in c.lower() or 'año' in c.lower()][0]

cols_tabla_proy = ['UBIGEO', col_dist_proy, col_anio_proy, col_pob_proy]
tabla_proy = df_proy[cols_tabla_proy].copy()
tabla_proy.columns = ['UBIGEO', 'Distrito', 'Año', 'Población Proyectada']

# 6. Despliegue en pantalla de ambas tablas
print("TABLA DEMOGRÁFICA CENSO 2017")
display(tabla_2017.round(2))

print("\nTABLA DEMOGRÁFICA PROYECCIONES POBLACIONALES (2018–2024)")
display(tabla_proy)

In [ ]:
def calcular_estadisticas_descriptivas(df, columnas):
    desc = df[columnas].describe(percentiles=[.25, .5, .75, .95]).T
    stats = desc[['count', 'min', '25%', '50%', 'mean', '75%', '95%', 'max']].copy()
    stats.columns = ['n', 'min', 'p25', 'mediana', 'media', 'p75', 'p95', 'max']
    return stats.round(2)

# 1. Columnas numéricas específicas para Censo 2017
cols_censo_stats = [
    'POBLACION_CENSADA_2017',
    'EDAD_0_4_2017',
    'EDAD_65_MAS_2017',
    'PCT_0_4',
    'PCT_65_MAS'
]
cols_censo_existentes = [c for c in cols_censo_stats if c in df_censo.columns]
descriptive_stats_censo = calcular_estadisticas_descriptivas(df_censo, cols_censo_existentes)

# 2. Columnas numéricas específicas para Proyecciones 2018–2024
cols_proy_stats = ['POBLACION_PROYECTADA']
descriptive_stats_proy = calcular_estadisticas_descriptivas(df_proy, cols_proy_stats)

# 3. Salida por pantalla (impresión directa con display)
print("Estadísticas Descriptivas - Censo 2017")
display(descriptive_stats_censo)

print("\nEstadísticas Descriptivas - Proyecciones 2018-2024")
display(descriptive_stats_proy)

### Controles técnicos para explorar las fuentes

Se comparan valores antes y después y se registra cada regla aplicada. La limpieza concluye antes de empezar los gráficos del EDA.


#### Controles. Decisiones de limpieza y archivos auditables


In [ ]:

if duplicados_hora or duplicados_meteo:
    raise ValueError("Hay horas duplicadas; inspeccionar conflictos.")
if duplicados_id:
    print("REVISAR: hay ID repetidos. La unión usa estación/hora, no ID.")

hora_lima = (pl.col("FECHA_HORA_ORIGINAL").dt.replace_time_zone("UTC")
             .dt.convert_time_zone("America/Lima")
             .dt.replace_time_zone(None))

aire_limpio = (
    tipados
    .filter(pl.col("ESTACION").is_not_null() &
            pl.col("FECHA_HORA_ORIGINAL").is_not_null())
    .with_columns(
        [pl.col("ESTACION").str.strip_chars().alias("ESTACION"),
         hora_lima.cast(pl.Datetime("us")).alias("FECHA_HORA_LIMA")]
        + [
            pl.when(pl.col(c + "_NUM") < 0).then(None)
              .otherwise(pl.col(c + "_NUM")).alias(c)
            for c in CONTAMINANTES
        ]
        + [pl.col(c + "_NUM").alias(c) for c in ["LATITUD", "LONGITUD", "ALTITUD"]]
    )
    .select([*encabezado, "FECHA_HORA_ORIGINAL", "FECHA_HORA_LIMA"])
)

meteo_limpio = meteo_raw.with_columns(
    pl.col("ESTACION").cast(pl.String).str.strip_chars().alias("ESTACION"),
    pl.col("FECHA_HORA_LIMA").cast(pl.Datetime("us")).alias("FECHA_HORA_LIMA"),
    *[
        pl.when(condicion).then(None).otherwise(pl.col(c))
          .cast(pl.Float64, strict=False).alias(c)
        for c, condicion in {
            "relative_humidity_2m": (pl.col("relative_humidity_2m") < 0) |
                                    (pl.col("relative_humidity_2m") > 100),
            "precipitation": pl.col("precipitation") < 0,
            "wind_speed_10m": pl.col("wind_speed_10m") < 0,
            "wind_direction_10m": (pl.col("wind_direction_10m") < 0) |
                                  (pl.col("wind_direction_10m") > 360),
            "pressure_msl": pl.col("pressure_msl") <= 0,
            "boundary_layer_height": pl.col("boundary_layer_height") < 0,
        }.items()
    ]
).select(["ESTACION", "FECHA_HORA_LIMA", *METEOROLOGIA])

RUTA_AIRE_LIMPIO = SALIDA / "aire_limpio.parquet"
RUTA_CLIMA_LIMPIO = SALIDA / "clima_limpio.parquet"
aire_limpio.sink_parquet(str(RUTA_AIRE_LIMPIO), compression="zstd")
meteo_limpio.sink_parquet(str(RUTA_CLIMA_LIMPIO), compression="zstd")
aire = pl.scan_parquet(str(RUTA_AIRE_LIMPIO))
clima = pl.scan_parquet(str(RUTA_CLIMA_LIMPIO))

posterior = aire.select(
    pl.len().alias("filas"),
    *[pl.col(c).null_count().alias("nulos_" + c) for c in CONTAMINANTES]
).collect(engine="streaming")
print("Filas originales:", n_aire_original)
print("Filas tras limpiar claves:", posterior["filas"][0])
print("Filas excluidas por clave inválida:", n_aire_original - posterior["filas"][0])
mostrar(posterior, "Resumen base 12 — Balance posterior a la limpieza")


inc = incidencias.row(0, named=True)
met = anomalías_meteo.row(0, named=True)
bitacora = pl.DataFrame([
    {"problema": "Fecha/estación ilegible", "casos_detectados":
     n_aire_original - posterior["filas"][0],
     "tecnica": "Excluir solo filas sin clave utilizable; conservar el CSV original"},
    {"problema": "Duplicados estación/hora", "casos_detectados": duplicados_hora,
     "tecnica": "Detener y revisar; no elegir arbitrariamente una lectura"},
    {"problema": "Concentración negativa", "casos_detectados":
     sum(int(inc["concentraciones_negativas_" + c]) for c in CONTAMINANTES),
     "tecnica": "Sustituir valor imposible por null; conservar hora/estación"},
    {"problema": "Meteorología físicamente imposible", "casos_detectados":
     sum(int(met[k]) for k in met),
     "tecnica": "Sustituir solo el valor imposible por null; registrar variable"},
    {"problema": "PM2.5 sin medición", "casos_detectados":
     int(posterior["nulos_PM2_5"][0]),
     "tecnica": "Conservar como null; cero NO significa ausencia de lectura"},
    {"problema": "PM2.5 positivo muy alto", "casos_detectados": None,
     "tecnica": "Cuantificar por estación en el EDA; conservar hasta investigar"},
])
mostrar(bitacora, "Resumen base 12A — Bitácora de problemas y técnica de tratamiento")
antes_aire = tipados.select(
    *[pl.col(c + "_NUM").null_count().alias(c) for c in CONTAMINANTES]
).collect(engine="streaming").row(0, named=True)
comparacion_aire = pl.DataFrame([
    {"variable": c, "nulos_antes": antes_aire[c],
     "nulos_despues": posterior["nulos_" + c][0]}
    for c in CONTAMINANTES
])
mostrar(comparacion_aire, "Resumen base 12B — Comprobación de faltantes antes/después")
print("Se preservaron los faltantes de PM2.5; no se aplicó imputación.")
print("Controles para EDA terminados; la preparación del modelo se documenta en la fase 3.")

aire = aire.select(["ESTACION", "FECHA_HORA_LIMA", "PM2_5"])
print("Columnas de aire que pasan al EDA y al cruce climático:", aire.collect_schema().names())


### Limpieza de la data de población

In [ ]:
# 1. Carga de datos
df_censo = pd.read_csv(RUTA_DEMO_CENSO)
df_proy = pd.read_csv(RUTA_DEMO_PROY)

# 2. Estandarización de texto (Llave primaria de distritos)
for df_temp in [df_censo, df_proy]:
    col_dist = [c for c in df_temp.columns if 'dist' in c.lower() or 'estac' in c.lower()][0]
    df_temp[col_dist] = (
        df_temp[col_dist]
        .astype(str)
        .str.replace('CAMPO_DE_MARTE', 'JESUS MARIA')
        .str.replace('_', ' ')
        .str.title()
        .str.strip()
    )

col_dist_censo = [c for c in df_censo.columns if 'dist' in c.lower() or 'estac' in c.lower()][0]
col_dist_proy = [c for c in df_proy.columns if 'dist' in c.lower() or 'estac' in c.lower()][0]

# 3. Auditoría de Valores Nulos y Duplicados
nulos_censo = df_censo.isna().sum().sum()
nulos_proy = df_proy.isna().sum().sum()

dup_censo = df_censo.duplicated(subset=[col_dist_censo]).sum()
dup_proy = df_proy.duplicated(subset=[col_dist_proy, 'ANIO']).sum() if 'ANIO' in df_proy.columns else 0

# 4. Control de Calidad y Detección de Valores Atípicos/Imposibles
col_pob_censo = [c for c in df_censo.columns if 'total' in c.lower() or 'pob' in c.lower()][0]
col_pob_proy = [c for c in df_proy.columns if 'pob' in c.lower()][0]

# Check A: Poblaciones negativas o de valor cero
pob_neg_censo = (df_censo[col_pob_censo] <= 0).sum()
pob_neg_proy = (df_proy[col_pob_proy] <= 0).sum()

# Check B: Inconsistencias lógicas de edad (Niños + Adultos Mayores > Población Total)
col_ninos = [c for c in df_censo.columns if ('0' in c or '4' in c) and c != col_pob_censo][0]
col_adultos = [c for c in df_censo.columns if ('65' in c or 'mas' in c.lower()) and c != col_pob_censo][0]

inconsistencia_edad = (
    (df_censo[col_ninos] + df_censo[col_adultos]) > df_censo[col_pob_censo]
).sum()

# Check C: Rangos de años fuera de periodo esperado en proyecciones (2018–2026)
anios_invalidos = (
    ((df_proy['ANIO'] < 2018) | (df_proy['ANIO'] > 2026)).sum()
    if 'ANIO' in df_proy.columns else 0
)

# 5. Generación de Bitácora Técnica de Diagnóstico
bitacora_demografica = pd.DataFrame([
    {
        "Dataset": "Censo 2017",
        "Registros": len(df_censo),
        "Valores Nulos": nulos_censo,
        "Duplicados (Distrito)": dup_censo,
        "Poblaciones <= 0": pob_neg_censo,
        "Inconsistencias Lógicas": inconsistencia_edad,
        "Acción / Estado": "Aprobado (Data limpia)"
    },
    {
        "Dataset": "Proyecciones Poblacionales",
        "Registros": len(df_proy),
        "Valores Nulos": nulos_proy,
        "Duplicados (Distrito-Año)": dup_proy,
        "Poblaciones <= 0": pob_neg_proy,
        "Inconsistencias Lógicas": anios_invalidos,
        "Acción / Estado": "Aprobado (Data limpia)"
    }
])

# 6. Exportación de data validada a Parquet
df_censo.to_parquet(SALIDA / "demografia_censo_limpio.parquet", index=False)
df_proy.to_parquet(SALIDA / "demografia_proyectada_limpia.parquet", index=False)

# 7. Salida de auditoría
mostrar(bitacora_demografica, "Control de Calidad y Auditoría de Fuentes Demográficas")

### Exploración de las fuentes originales

Primero se explora el aire preparado, después el clima preparado, luego ambas fuentes unidas por estación y hora y por utlimo se explora la data de población y proyección.





### EDA del aire


### Cobertura y diferencias entre estaciones

¿cuánto mide realmente cada estación por año y cómo cambian mediana, cuartiles, desviación y máximos?


In [ ]:
perfil = (
    aire.group_by("ESTACION")
    .agg(
        pl.len().alias("horas_totales"),
        pl.col("PM2_5").count().alias("horas_pm25"),
        pl.col("PM2_5").mean().alias("media_pm25"),
        pl.col("PM2_5").median().alias("mediana_pm25"),
        pl.col("PM2_5").std().alias("desv_pm25"),
        pl.col("PM2_5").min().alias("min_pm25"),
        pl.col("PM2_5").quantile(.25, "linear").alias("q1_pm25"),
        pl.col("PM2_5").quantile(.75, "linear").alias("q3_pm25"),
        pl.col("PM2_5").quantile(.95, "linear").alias("p95_pm25"),
        pl.col("PM2_5").max().alias("max_pm25"),
        pl.col("FECHA_HORA_LIMA").min().alias("inicio"),
        pl.col("FECHA_HORA_LIMA").max().alias("fin"),
    )
    .with_columns((100 * pl.col("horas_pm25") / pl.col("horas_totales"))
                  .round(2).alias("cobertura_pct"))
    .sort("mediana_pm25", descending=True)
    .collect(engine="streaming")
)
mostrar(perfil, "Resumen base 13 — Comparación de estaciones: centro y cobertura")

anual = (
    aire.with_columns(pl.col("FECHA_HORA_LIMA").dt.year().alias("ANIO"))
    .group_by(["ESTACION", "ANIO"])
    .agg(pl.len().alias("filas"),
         pl.col("PM2_5").count().alias("medidas"),
         pl.col("PM2_5").median().alias("mediana"))
    .with_columns((100 * pl.col("medidas") / pl.col("filas"))
                  .round(1).alias("cobertura_pct"))
    .sort(["ESTACION", "ANIO"]).collect(engine="streaming")
)
mostrar(anual, "Resumen base 14 — Cobertura y mediana de PM2.5 por estación y año", 100)
saltos = (
    aire.sort(["ESTACION", "FECHA_HORA_LIMA"])
    .with_columns(pl.col("FECHA_HORA_LIMA").diff().over("ESTACION").alias("intervalo"))
    .group_by("intervalo").agg(pl.len().alias("apariciones"))
    .sort("apariciones", descending=True).head(10).collect()
)
mostrar(saltos, "Resumen base 14A — Distancias temporales entre filas consecutivas por estación")
pd_anual = anual.to_pandas().pivot(index="ESTACION", columns="ANIO",
                                    values="cobertura_pct")
plt.figure(figsize=(11, 5))
plt.imshow(pd_anual, aspect="auto", cmap="YlGnBu", vmin=0, vmax=100)
plt.colorbar(label="Horas con PM2.5 (%)")
plt.yticks(range(len(pd_anual)), pd_anual.index)
plt.xticks(range(len(pd_anual.columns)), pd_anual.columns)
plt.title("Disponibilidad de PM2.5 por estación y año (2024 parcial)")
figura("fig01_cobertura_estacion_anio.png")




### Duración de las ausencias de PM2.5



In [ ]:

anual_global = (
    aire.with_columns(pl.col("FECHA_HORA_LIMA").dt.year().alias("ANIO"))
    .group_by("ANIO")
    .agg(pl.len().alias("horas_en_archivo"),
         pl.col("PM2_5").count().alias("horas_con_pm25"))
    .with_columns((100 * (1 - pl.col("horas_con_pm25") /
                            pl.col("horas_en_archivo"))).round(2).alias("faltantes_pct"))
    .sort("ANIO").collect(engine="streaming")
)
mostrar(anual_global, "Resumen base 14B — Faltantes de PM2.5 por año (2024 es parcial)")

import pandas as pd
marcas = (
    aire.select("ESTACION", "FECHA_HORA_LIMA", "PM2_5")
    .collect(engine="in-memory").to_pandas()
    .sort_values(["ESTACION", "FECHA_HORA_LIMA"], kind="stable")
    .reset_index(drop=True)
)
sin_pm25 = marcas["PM2_5"].isna()
inicio_bloque = (
    marcas["ESTACION"].ne(marcas["ESTACION"].shift()) |
    marcas["FECHA_HORA_LIMA"].diff().ne(pd.Timedelta(hours=1)) |
    sin_pm25.ne(sin_pm25.shift(fill_value=False))
)
marcas["bloque"] = inicio_bloque.cumsum()
rachas_pd = (
    marcas.loc[sin_pm25]
    .groupby(["ESTACION", "bloque"], as_index=False, observed=True)
    .agg(horas=("PM2_5", "size"),
         inicio=("FECHA_HORA_LIMA", "min"),
         fin=("FECHA_HORA_LIMA", "max"))
    .sort_values("horas", ascending=False)
)
rachas = pl.from_pandas(rachas_pd)
resumen_rachas = (
    rachas.group_by("ESTACION")
    .agg(pl.len().alias("interrupciones"),
         pl.col("horas").sum().alias("horas_sin_pm25"),
         pl.col("horas").max().alias("interrupcion_maxima_h"),
         (pl.col("horas") >= 24).sum().alias("interrupciones_24h_o_mas"),
         (pl.col("horas") >= 168).sum().alias("interrupciones_7d_o_mas"),
         pl.col("horas").filter(pl.col("horas") >= 168).sum()
           .alias("horas_en_interrupciones_7d_o_mas"))
    .sort("horas_sin_pm25", descending=True)
)
mostrar(resumen_rachas, "Resumen base 14C — Interrupciones de PM2.5 por estación")
mostrar(rachas.head(12).drop("bloque"),
        "Resumen base 14D — Las doce interrupciones más largas")
nulos_en_rachas_largas = int(rachas.filter(pl.col("horas") >= 168)["horas"].sum())
total_nulos = int(posterior["nulos_PM2_5"][0])
print("Proporción de horas sin PM2.5 en interrupciones de >=7 días:",
      round(100 * nulos_en_rachas_largas / total_nulos, 2), "%")

sin_pm25_por_hora = (
    marcas.assign(sin_pm25=sin_pm25)
    .groupby("FECHA_HORA_LIMA", sort=True)["sin_pm25"].sum()
)
print("Horas en que faltó PM2.5 simultáneamente en las siete estaciones:",
      int((sin_pm25_por_hora == 7).sum()))

plt.figure(figsize=(10, 4))
plt.bar(anual_global["ANIO"].to_numpy().astype(str),
        anual_global["faltantes_pct"].to_numpy(), color="#487caa")
plt.ylabel("Horas sin PM2.5 (%)")
plt.xlabel("Año del archivo (2024 parcial)")
plt.title("Ausencias por año: se incluyen todas las estaciones y horas")
figura("fig01b_faltantes_pm25_por_anio.png")

principales = rachas.head(10).to_pandas().iloc[::-1]
plt.figure(figsize=(11, 5))
etiquetas = principales["ESTACION"] + " · " + principales["inicio"].dt.strftime("%Y-%m-%d")
plt.barh(etiquetas, principales["horas"] / 24, color="#bb7653")
plt.xlabel("Duración del intervalo sin PM2.5 (días)")
plt.title("Diez interrupciones de medición más largas")
figura("fig01c_interrupciones_pm25.png")


### Distribución de PM2.5 y boxplots



In [ ]:
est = aire.select(
    pl.col("PM2_5").count().alias("validas"),
    pl.col("PM2_5").mean().alias("media"),
    pl.col("PM2_5").median().alias("mediana"),
    pl.col("PM2_5").quantile(.25, "linear").alias("q1"),
    pl.col("PM2_5").quantile(.75, "linear").alias("q3"),
    pl.col("PM2_5").quantile(.95, "linear").alias("p95"),
    pl.col("PM2_5").max().alias("maximo")
).collect(engine="streaming")
mostrar(est, "Resumen base 15 — Distribución PM2.5 después de limpieza")
q1, q3 = est["q1"][0], est["q3"][0]
limite_iqr = q3 + 1.5 * (q3 - q1)
print(f"Límite IQR superior: {limite_iqr:.2f} (bandera de inspección, NO eliminación)")

# Agregamos las 363 mil mediciones en intervalos de amplitud 5.
hist = (
    aire.filter(pl.col("PM2_5").is_not_null())
    .group_by(((pl.col("PM2_5") // 5).clip(0, 40)).alias("tramo"))
    .agg(pl.len().alias("conteo"))
    .sort("tramo").collect(engine="streaming")
)
mostrar(hist, "Resumen base 16 — Histograma tabular de todas las mediciones", 45)
plt.figure(figsize=(12, 4))
plt.bar(hist["tramo"].to_numpy() * 5, hist["conteo"].to_numpy(), width=4.5)
plt.xlabel("Inicio de tramo PM2.5 (>=200 agrupado en un solo tramo)")
plt.ylabel("Número de horas")
plt.title("Distribución completa de PM2.5 (sin tomar muestra)")
figura("fig02_histograma_completo_pm25.png")

pm_box = (
    aire.select(["ESTACION", "PM2_5"])
    .filter(pl.col("PM2_5").is_not_null())
    .collect(engine="in-memory").to_pandas()
)
p99_visual = pm_box["PM2_5"].quantile(.99)
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
estilo_picos = {"marker": ".", "markersize": 1.4, "alpha": .25}
for ax in axes:
    sns.boxplot(data=pm_box, y="ESTACION", x="PM2_5", ax=ax,
                showfliers=True, flierprops=estilo_picos, color="#74a9cf")
    ax.set_ylabel("Estación")
axes[0].set_xscale("log")
axes[0].set_title("Todos los valores, escala logarítmica")
axes[1].set_xlim(0, p99_visual)
axes[1].set_title("Ampliación hasta p99")
fig.suptitle("Boxplots PM2.5: puntos fuera de bigotes ≠ errores confirmados")
figura("fig02b_boxplots_pm25_estaciones.png")
print(f"El gráfico derecho termina en p99={p99_visual:.2f}; no se eliminaron filas.")
del pm_box




### Episodios y revisión de atípicos


In [ ]:

umbrales_iqr = perfil.select(
    "ESTACION", "horas_pm25", "q1_pm25", "q3_pm25"
).with_columns(
    (pl.col("q3_pm25") - pl.col("q1_pm25")).alias("iqr")
).with_columns(
    (pl.col("q1_pm25") - 1.5 * pl.col("iqr")).alias("umbral_inferior"),
    (pl.col("q3_pm25") + 1.5 * pl.col("iqr")).alias("umbral_superior")
)
mostrar(umbrales_iqr, "Resumen base 17 — Técnica IQR y umbrales por estación")

banderas_df = (
    aire.select(["ESTACION", "FECHA_HORA_LIMA", "PM2_5"])
        .collect(engine="in-memory")
        .join(umbrales_iqr.select("ESTACION", "umbral_inferior",
                                    "umbral_superior"),
              on="ESTACION", how="left", validate="m:1")
        .with_columns(
            (pl.col("PM2_5") > pl.col("umbral_superior"))
                .fill_null(False).alias("atipico_alto_iqr"),
            (pl.col("PM2_5") < pl.col("umbral_inferior"))
                .fill_null(False).alias("atipico_bajo_iqr"),
        )
)
extremos = (
    banderas_df.group_by("ESTACION").agg(
        pl.col("PM2_5").count().alias("horas_validas"),
        pl.col("atipico_alto_iqr").sum().alias("atipicos_altos"),
        pl.col("atipico_bajo_iqr").sum().alias("atipicos_bajos"),
    ).with_columns(
        (100 * pl.col("atipicos_altos") / pl.col("horas_validas"))
            .round(2).alias("atipicos_altos_pct")
    ).sort("atipicos_altos", descending=True)
)
mostrar(extremos, "Resumen base 18 — Recuento de alertas para revisar por estación")
print("Atípicos altos IQR (total):", int(extremos["atipicos_altos"].sum()))
mostrar(
    banderas_df.filter(pl.col("atipico_alto_iqr"))
        .select("ESTACION", "FECHA_HORA_LIMA", "PM2_5")
        .sort("PM2_5", descending=True).head(15),
    "Resumen base 19 — Mediciones altas a investigar"
)

vecinos = (
    banderas_df.sort(["ESTACION", "FECHA_HORA_LIMA"])
        .with_columns(
            pl.col("PM2_5").shift(1).over("ESTACION").alias("_pm_anterior"),
            pl.col("PM2_5").shift(-1).over("ESTACION").alias("_pm_siguiente"),
            pl.col("FECHA_HORA_LIMA").shift(1).over("ESTACION").alias("_hora_anterior"),
            pl.col("FECHA_HORA_LIMA").shift(-1).over("ESTACION").alias("_hora_siguiente"),
        )
)
revisar_picos = vecinos.filter(
    pl.col("atipico_alto_iqr") &
    (pl.col("_hora_anterior") == pl.col("FECHA_HORA_LIMA") - pl.duration(hours=1)) &
    (pl.col("_hora_siguiente") == pl.col("FECHA_HORA_LIMA") + pl.duration(hours=1)) &
    pl.col("_pm_anterior").is_not_null() &
    pl.col("_pm_siguiente").is_not_null() &
    (pl.col("PM2_5") > 5 * pl.max_horizontal("_pm_anterior", "_pm_siguiente"))
)
print("Picos aislados que merecen revisión:", revisar_picos.height)
mostrar(
    revisar_picos.select("ESTACION", "FECHA_HORA_LIMA", "PM2_5",
                         "_pm_anterior", "_pm_siguiente")
                 .sort("PM2_5", descending=True).head(15),
    "Resumen base 20 — Candidatos para contrastar con la fuente y horas vecinas"
)
RUTA_BANDERAS = SALIDA / "banderas_revision_pm25.parquet"
banderas_df.write_parquet(str(RUTA_BANDERAS), compression="zstd")
print("Decisión: se marcaron posibles atípicos; se eliminaron 0 picos positivos.")

episodios = (
    aire.with_columns(pl.col("FECHA_HORA_LIMA").dt.date().alias("DIA"))
    .group_by(["ESTACION", "DIA"])
    .agg(pl.col("PM2_5").count().alias("horas_validas"),
         pl.col("PM2_5").mean().alias("media_dia"),
         pl.col("PM2_5").max().alias("max_horario"))
    .filter(pl.col("horas_validas") >= 18)
    .sort("media_dia", descending=True).head(10).collect()
)
mostrar(episodios, "Resumen base 21 — Diez días con mayor media observada (>=18 h)")
if episodios.height:
    plt.figure(figsize=(11, 4))
    etiquetas = [f"{e} {d}" for e, d in
                 zip(episodios["ESTACION"], episodios["DIA"])]
    plt.barh(etiquetas[::-1], episodios["media_dia"].to_numpy()[::-1])
    plt.xlabel("Media observada de PM2.5 en el día")
    plt.title("Días más altos con al menos 18 horas medidas")
    figura("fig03_top_dias_pm25.png")


### Tendencias y perfil horario

¿se observan cambios por mes y hora? El cuadro conserva todos los meses y su cobertura; las líneas del gráfico mensual solo se dibujan en meses con al menos 75 % de horas medidas. El perfil horario se recalculó usando hora Perú, según la referencia indicada por el equipo.


In [ ]:
mensual = (
    aire.with_columns(pl.col("FECHA_HORA_LIMA").dt.truncate("1mo").alias("MES"))
    .group_by(["ESTACION", "MES"])
    .agg(pl.col("PM2_5").median().alias("mediana"),
         pl.col("PM2_5").count().alias("validas"),
         pl.len().alias("total"))
    .with_columns((100 * pl.col("validas") / pl.col("total"))
                  .round(1).alias("cobertura_pct"))
    .sort(["ESTACION", "MES"]).collect(engine="streaming")
)
mostrar(mensual, "Resumen base 20 — Estadísticos mensuales", 15)
plt.figure(figsize=(13, 5))
for s, sub in mensual.to_pandas().groupby("ESTACION"):
    # Meses con menos del 75 % de mediciones no sustentan la misma lectura.
    plt.plot(sub["MES"], sub["mediana"].where(sub["cobertura_pct"] >= 75),
             label=s, linewidth=1.2)
plt.legend(bbox_to_anchor=(1, 1), loc="upper left", title="Estación")
plt.xlabel("Mes"); plt.ylabel("Mediana de PM2.5")
plt.title("Tendencia mensual (líneas solo para meses con cobertura >=75 %)")
figura("fig04_tendencia_mensual.png")

hora = (
    aire.with_columns(pl.col("FECHA_HORA_LIMA").dt.hour().alias("HORA_LIMA"))
    .group_by(["ESTACION", "HORA_LIMA"])
    .agg(pl.col("PM2_5").median().alias("mediana"),
         pl.col("PM2_5").count().alias("observaciones"))
    .sort(["ESTACION", "HORA_LIMA"]).collect(engine="streaming")
)
mostrar(hora, "Resumen base 21 — Perfil horario por estación", 25)
plt.figure(figsize=(12, 5))
for s, sub in hora.to_pandas().groupby("ESTACION"):
    plt.plot(sub["HORA_LIMA"], sub["mediana"], label=s)
plt.xticks(range(0, 24, 2))
plt.xlabel("Hora oficial del Perú")
plt.ylabel("Mediana de PM2.5")
plt.legend(bbox_to_anchor=(1, 1), loc="upper left", title="Estación")
plt.title("Perfil horario")
figura("fig05_perfil_horario.png")




### EDA del clima


### Distribuciones, estacionalidad y cobertura climática

Se examinan por separado los rangos y boxplots del clima, las medianas por mes y los faltantes de altura de la capa límite por año. Aún no se cruza PM2.5 con el clima en esta subsección.


In [ ]:
clima_descriptivo = []
for c in METEOROLOGIA:
    x = pl.col(c)
    r = clima.select(
        pl.len().alias("filas"), x.count().alias("no_nulos"),
        x.min().alias("min"), x.median().alias("mediana"),
        x.mean().alias("media"), x.quantile(.95, "linear").alias("p95"),
        x.max().alias("max")
    ).collect(engine="streaming").row(0, named=True)
    clima_descriptivo.append({"variable": c, **r,
                             "faltantes_pct": round(100*(r["filas"]-r["no_nulos"])/r["filas"], 3)})
clima_tabla = pl.DataFrame(clima_descriptivo)
mostrar(clima_tabla, "Resumen base 22 — Clima limpio: centro, rango y valores faltantes")


clima_anual = (
    clima.with_columns(pl.col("FECHA_HORA_LIMA").dt.year().alias("ANIO"))
    .group_by("ANIO")
    .agg(pl.len().alias("horas_clima"),
         pl.col("boundary_layer_height").count().alias("horas_con_capa_limite"))
    .with_columns((100 * (1 - pl.col("horas_con_capa_limite") /
                            pl.col("horas_clima"))).round(2).alias("capa_limite_faltante_pct"))
    .sort("ANIO").collect(engine="streaming")
)
mostrar(clima_anual,
        "Resumen base 22A — Faltantes de altura de capa límite por año (2024 parcial)")
plt.figure(figsize=(10, 4))
plt.bar(clima_anual["ANIO"].to_numpy().astype(str),
        clima_anual["capa_limite_faltante_pct"].to_numpy(), color="#87618f")
plt.ylabel("Altura de la capa límite sin dato (%)")
plt.xlabel("Año (2024 parcial)")
plt.title("Cobertura de la capa límite: el porcentaje global oculta 2024")
figura("fig06c_faltantes_capa_limite_por_anio.png")


clima_por_mes = (
    clima.with_columns(pl.col("FECHA_HORA_LIMA").dt.month().alias("MES_NUM"))
    .group_by("MES_NUM")
    .agg(pl.len().alias("horas"),
         pl.col("temperature_2m").median().alias("temperatura_mediana"),
         pl.col("relative_humidity_2m").median().alias("humedad_mediana"),
         pl.col("wind_speed_10m").median().alias("viento_mediana"))
    .sort("MES_NUM").collect(engine="streaming")
)
mostrar(clima_por_mes, "Resumen base 22B — Perfil meteorológico por mes calendario")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, campo, titulo in [
    (axes[0], "temperatura_mediana", "Temperatura: mediana por mes"),
    (axes[1], "humedad_mediana", "Humedad relativa: mediana por mes"),
]:
    ax.plot(clima_por_mes["MES_NUM"], clima_por_mes[campo], marker="o")
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("Mes calendario")
    ax.set_ylabel("Mediana de la variable")
    ax.set_title(titulo)
fig.suptitle("EDA meteorológico independiente, sobre todas las horas disponibles")
figura("fig06d_perfil_mensual_clima.png")

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
seleccion = [
    ("temperature_2m", 2), ("relative_humidity_2m", 5),
    ("wind_speed_10m", 1), ("precipitation", 1),
]
for ax, (c, paso) in zip(axes.flat, seleccion):
    tabla = (
        clima.filter(pl.col(c).is_not_null())
        .group_by(((pl.col(c) / paso).floor() * paso).alias("tramo"))
        .agg(pl.len().alias("n")).sort("tramo").collect(engine="streaming")
    )

    x = tabla["tramo"].to_numpy(); y = tabla["n"].to_numpy()
    if len(x):
        hasta = np.quantile(x, .99)
        ax.bar(x[x <= hasta], y[x <= hasta], width=paso * .9)
    ax.set_title(c); ax.set_xlabel("Valor de la fuente"); ax.set_ylabel("Horas")
fig.suptitle("Meteorología: frecuencias agregadas sobre el conjunto completo")
figura("fig06_distribuciones_clima.png")

clima_box = clima.select([
    "ESTACION", "temperature_2m", "relative_humidity_2m", "wind_speed_10m"
]).collect(engine="in-memory").to_pandas()
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
for ax, variable in zip(axes, ["temperature_2m", "relative_humidity_2m",
                               "wind_speed_10m"]):
    sns.boxplot(data=clima_box, y="ESTACION", x=variable, ax=ax,
                showfliers=True,
                flierprops={"marker": ".", "markersize": 1, "alpha": .20},
                color="#84b6a0")
    ax.set_title(variable)
    ax.set_ylabel("Estación" if ax is axes[0] else "")
fig.suptitle("Boxplots meteorológicos; valor raro no equivale a error")
figura("fig06b_boxplots_clima.png")
del clima_box




### EDA conjunto: aire y clima


### Unión y comprobación de claves

Se cruza por **estación y hora local**. La unión se materializa una vez, se guarda como Parquet y se verifica que no aparezcan filas adicionales. La referencia horaria se adoptó a partir de lo indicado por el equipo; la coincidencia de claves es un control adicional de unión.


In [ ]:

aire_union = aire.collect(engine="in-memory")
clima_union = clima.select(["ESTACION", "FECHA_HORA_LIMA", *METEOROLOGIA]).collect(
    engine="in-memory"
)

aire_union = aire_union.with_columns(
    pl.col("ESTACION").cast(pl.String),
    pl.col("FECHA_HORA_LIMA").cast(pl.Datetime("us"))
)
clima_union = clima_union.with_columns(
    pl.col("ESTACION").cast(pl.String),
    pl.col("FECHA_HORA_LIMA").cast(pl.Datetime("us")),
    pl.lit(True).alias("_match_clima")
)
print("Claves aire:", aire_union.schema["ESTACION"],
      aire_union.schema["FECHA_HORA_LIMA"])
print("Claves clima:", clima_union.schema["ESTACION"],
      clima_union.schema["FECHA_HORA_LIMA"])
unido_df = aire_union.join(
    clima_union, on=["ESTACION", "FECHA_HORA_LIMA"],
    how="left", validate="1:1"
)
RUTA_UNIDO = SALIDA / "aire_clima_unido.parquet"
unido_df.write_parquet(str(RUTA_UNIDO), compression="zstd")
del aire_union, clima_union, unido_df
unido = pl.scan_parquet(str(RUTA_UNIDO))
join_resumen = unido.select(
    pl.len().alias("filas_resultado"),
    pl.col("_match_clima").count().alias("claves_emparejadas"),
    (pl.col("PM2_5").is_not_null() &
     pl.col("_match_clima").is_not_null()).sum().alias("pm25_y_clima"),
    *[pl.col(c).null_count().alias("nulos_" + c) for c in METEOROLOGIA],
).collect(engine="in-memory")
mostrar(join_resumen, "Resumen base 23 — Calidad de la unión aire-clima")
assert join_resumen["filas_resultado"][0] == posterior["filas"][0]
print("Coincidencia de claves (%):",
      round(100 * join_resumen["claves_emparejadas"][0] /
            join_resumen["filas_resultado"][0], 2))
print("Una coincidencia de fechas NO prueba la zona horaria.")

### Correlaciones y relaciones exploratorias

¿qué variables se relacionan con PM2.5?, ¿cuántos pares respaldan cada correlación?, ¿cambia por estación? Hay tabla de pares, barras, mapa de calor y densidad humedad–PM2.5. Recordemos que una correlación no implica causalidad


In [ ]:

angulo = pl.col("wind_direction_10m") * math.pi / 180
unido = unido.with_columns(
    angulo.sin().alias("viento_seno"),
    angulo.cos().alias("viento_coseno")
)
columnas_corr = ["temperature_2m", "relative_humidity_2m",
                 "precipitation", "pressure_msl", "wind_speed_10m",
                 "boundary_layer_height", "viento_seno", "viento_coseno"]
filas_corr = []
for c in columnas_corr:
    par = unido.filter(pl.col("PM2_5").is_not_null() & pl.col(c).is_not_null())
    resultado = par.select(pl.len().alias("pares"),
                           pl.corr("PM2_5", c).alias("r")).collect(engine="streaming")
    filas_corr.append({"variable": c, "pares": resultado["pares"][0],
                       "r_pearson": resultado["r"][0]})
corr_tabla = pl.DataFrame(filas_corr)
mostrar(corr_tabla, "Resumen base 24 — Correlación PM2.5 por variable y pares disponibles")

plt.figure(figsize=(9, 5))
cp = corr_tabla.sort("r_pearson").to_pandas()
plt.barh(cp["variable"], cp["r_pearson"])
plt.axvline(0, color="black", linewidth=.8)
plt.xlabel("r de Pearson (ver tamaño de muestra en la tabla)")
plt.title("Correlaciones exploratorias")
figura("fig07_correlaciones_aire_clima.png")

vars_matriz = ["PM2_5", "temperature_2m",
               "relative_humidity_2m", "precipitation", "pressure_msl",
               "wind_speed_10m", "boundary_layer_height",
               "viento_seno", "viento_coseno"]
numericas_matriz = unido.select(vars_matriz).collect(engine="in-memory").to_pandas()
matriz = numericas_matriz.corr(method="pearson")  # pares disponibles por variable
plt.figure(figsize=(11, 9))
sns.heatmap(matriz, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True)
plt.title("Correlaciones entre contaminantes y meteorología")
figura("fig07b_heatmap_correlaciones.png")
del numericas_matriz

dispersion = (
    unido.select(["relative_humidity_2m", "PM2_5"])
         .filter(pl.col("relative_humidity_2m").is_not_null() &
                 pl.col("PM2_5").is_not_null())
         .collect(engine="in-memory").to_pandas()
)
plt.figure(figsize=(8, 5))
hb = plt.hexbin(dispersion["relative_humidity_2m"], dispersion["PM2_5"],
                gridsize=48, mincnt=1, bins="log", cmap="viridis")
plt.ylim(0, dispersion["PM2_5"].quantile(.99))
plt.colorbar(hb, label="Frecuencia (escala logarítmica)")
plt.xlabel("Humedad relativa (dato histórico)")
plt.ylabel("PM2.5 (vista hasta p99)")
plt.title("Densidad de observaciones: humedad y PM2.5")
figura("fig07c_densidad_humedad_pm25.png")
del dispersion

por_estacion = (
    unido.filter(pl.col("PM2_5").is_not_null() &
                  pl.col("wind_speed_10m").is_not_null())
    .group_by("ESTACION")
    .agg(pl.len().alias("pares"), pl.corr("PM2_5", "wind_speed_10m").alias("r_viento"))
    .sort("ESTACION").collect(engine="streaming")
)
mostrar(por_estacion, "Resumen base 25 — PM2.5 y viento por estación")
viento_tramos = (
    unido.filter(pl.col("PM2_5").is_not_null() &
                  pl.col("wind_speed_10m").is_not_null() &
                  (pl.col("wind_speed_10m") >= 0))
    .group_by(((pl.col("wind_speed_10m") // 2) * 2).clip(0, 10).alias("tramo_viento"))
    .agg(pl.col("PM2_5").median().alias("mediana_pm25"),
         pl.len().alias("pares"))
    .sort("tramo_viento").collect(engine="streaming")
)
mostrar(viento_tramos, "Resumen base 26 — PM2.5 por tramo de viento (último tramo >=10)")
plt.figure(figsize=(8, 4))
plt.bar(viento_tramos["tramo_viento"].to_numpy().astype(str),
        viento_tramos["mediana_pm25"].to_numpy())
plt.xlabel("Velocidad del viento, tramos de 2 (último >=10)")
plt.ylabel("Mediana de PM2.5")
plt.title("Relación descriptiva, sin atribuir causalidad")
figura("fig08_pm25_por_viento.png")


### EDA conjunto: proyección y censo

### Perfiles demográficos por distrito
Se comparan las proporciones de población de 0 a 4 años y de 65 años o más del Censo 2017. Son indicadores de contexto distrital; este análisis no mide exposición individual ni efectos clínicos.

In [ ]:
# 1. Transformación de datos para barras agrupadas
bar_data = pd.melt(
    tabla_2017,
    id_vars=['Distrito'],
    value_vars=['% 0-4 años', '% 65+ años'],
    var_name='Grupo Etario',
    value_name='Porcentaje (%)',
)

# 2. Configuración de figura y lienzo
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=bar_data,
    x='Distrito',
    y='Porcentaje (%)',
    hue='Grupo Etario',
    palette={'% 0-4 años': '#3498db', '% 65+ años': '#e74c3c'},
)

# 3. Personalización de ejes, títulos y leyenda
plt.title(
    'Comparativa de Porcentaje de Población Vulnerable por Distrito (Censo'
    ' 2017)',
    fontsize=14,
    fontweight='bold',
    pad=15,
)
plt.ylabel('Porcentaje de la Población Total (%)', fontsize=11)
plt.xlabel('Distrito', fontsize=11)
plt.xticks(rotation=25, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.ylim(0, bar_data['Porcentaje (%)'].max() * 1.18)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.legend(
    title='Grupo de Riesgo',
    title_fontsize='11',
    fontsize='10',
    loc='upper right',
)

# 4. Inserción de etiquetas numéricas sobre cada barra
for p in ax.patches:
  val = p.get_height()
  if not np.isnan(val) and val > 0:
    ax.annotate(
        f'{val:.1f}%',
        (p.get_x() + p.get_width() / 2.0, val),
        ha='center',
        va='bottom',
        xytext=(0, 4),
        textcoords='offset points',
        fontsize=9.5,
        fontweight='bold',
        color='#2c3e50',
    )

plt.tight_layout()
plt.savefig(SALIDA/'fig10_poblacion_vulnerable.png',dpi=150,bbox_inches='tight')
plt.show()

### Proyección de la población total por distrito (2018–2024)
El archivo contiene años hasta 2026. Para mantener el alcance del EDA se muestran las 49 observaciones distrito-año de 2018–2024. Las proyecciones son contexto demográfico y no se incorporan como predictores.

In [ ]:
# 1. Identificación de columnas y filtrado del periodo 2018-2024
col_dist_proy = [c for c in df_proy.columns if 'dist' in c.lower() or 'estac' in c.lower()][0]
col_anio_proy = [c for c in df_proy.columns if 'an' in c.lower() or 'año' in c.lower()][0]
col_pob_proy = [c for c in df_proy.columns if 'pob' in c.lower() or 'total' in c.lower()][0]

df_proy_filtrado = df_proy[
    (df_proy[col_anio_proy] >= 2018) & (df_proy[col_anio_proy] <= 2024)
].copy()

# 2. Configuración de figura y gráfico de líneas
plt.figure(figsize=(14, 7))
ax = sns.lineplot(
    data=df_proy_filtrado,
    x=col_anio_proy,
    y=col_pob_proy,
    hue=col_dist_proy,
    marker='o',
    linewidth=2.5,
    palette='tab10',
    errorbar=None
)

# 3. Personalización de ejes, títulos y leyenda
plt.title('Evolución de la Población Total Proyectada por Distrito (2018–2024)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Año', fontsize=11)
plt.ylabel('Población Total Proyectada', fontsize=11)
plt.xticks(sorted(df_proy_filtrado[col_anio_proy].unique()), fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)

# Desactivar notación científica en el eje Y
plt.ticklabel_format(style='plain', axis='y')

plt.legend(
    title='Distrito',
    title_fontsize='11',
    fontsize='9.5',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

# 4. Inserción de etiquetas con valores sobre los puntos
for line in ax.get_lines():
    x_data = line.get_xdata()
    y_data = line.get_ydata()
    for x, y in zip(x_data, y_data):
        if not pd.isna(y):
            ax.annotate(
                f'{int(y):,}',
                (x, y),
                textcoords="offset points",
                xytext=(0, 5),
                ha='center',
                fontsize=7.5,
                color=line.get_color()
            )

plt.tight_layout()
plt.savefig(SALIDA/'fig11_proyeccion_poblacion.png',dpi=150,bbox_inches='tight')
plt.show()

### 2.2. EDA de los datos complementarios

Se estudian por separado ERA5, IFS, hogares con vehículo y anomalías térmicas. ERA5 e IFS permanecen identificados como modelos diferentes. Las detecciones de NASA no equivalen a incendios independientes ni a concentración de PM2.5. El CSV combinado se usa para la unión; los archivos separados permiten comprobar su procedencia.

In [ ]:
import hashlib

def tabla_h2(df, numero, titulo, limite=40):
    df=pd.DataFrame(df).copy()
    print(f'\nTabla {numero}. {titulo}')
    display(df.head(limite).style.hide(axis='index').format(precision=3, na_rep='—'))
    df.to_csv(SALIDA/f'tabla_{numero:02d}.csv',index=False,encoding='utf-8-sig')
    return df

def fig_h2(numero, nombre):
    plt.tight_layout()
    plt.savefig(SALIDA/f'fig{numero:02d}_{nombre}.png',dpi=150,bbox_inches='tight')
    plt.show();plt.close()

RESUMEN_H2={}
comp=pd.read_csv(RUTA_COMPLEMENTOS,dtype={'UBIGEO':str},parse_dates=['FECHA_HORA_LIMA','FECHA_HORA_UTC'])
calor=pd.read_csv(RUTA_CALOR,dtype={'acq_time':str,'version':str},parse_dates=['FECHA_HORA_LIMA','FECHA_HORA_UTC'])
vehiculos=pd.read_csv(RUTA_VEHICULOS,dtype={'UBIGEO':str})
assert not comp.duplicated(['ESTACION','FECHA_HORA_LIMA']).any()
assert comp.FECHA_HORA_UTC.eq(comp.FECHA_HORA_LIMA+pd.Timedelta(hours=5)).all()
assert comp.ESTACION.nunique()==7 and len(comp)==613704
inventario_h2=tabla_h2([
 {'Fuente':'CSV complementario combinado','Filas':len(comp),'Columnas':len(comp.columns),'Periodo':'2015–2024','Unidad':'estación-hora'},
 {'Fuente':'ERA5 adicional (dentro del CSV)','Filas':len(comp),'Columnas':7,'Periodo':'2015–2024','Unidad':'estación-hora'},
 {'Fuente':'IFS adicional (dentro del CSV)','Filas':int(comp.ifs_disponible.sum()),'Columnas':7,'Periodo':'2017–2024','Unidad':'estación-hora'},
 {'Fuente':'Tenencia vehicular, 7 distritos','Filas':len(vehiculos),'Columnas':len(vehiculos.columns),'Periodo':'Censo 2017','Unidad':'hogares por distrito'},
 {'Fuente':'NASA FIRMS, Lima y entorno','Filas':len(calor),'Columnas':len(calor.columns),'Periodo':'archivos UTC 2015–2024','Unidad':'píxel térmico detectado'}],12,'Fuentes complementarias y unidades de análisis')
v_modelo=['wind_speed_100m','wind_direction_100m','shortwave_radiation','cloud_cover','cloud_cover_low','surface_pressure','dew_point_2m']
completitud_h2=tabla_h2([
 {'Variable':v,'ERA5 faltantes':int(comp['era5_'+v].isna().sum()),
  'IFS faltantes':int(comp['ifs_'+v].isna().sum()),
  'IFS % faltante':100*comp['ifs_'+v].isna().mean()} for v in v_modelo],13,'Completitud de las variables meteorológicas nuevas')
disponibilidad=comp.assign(anio=comp.FECHA_HORA_LIMA.dt.year).groupby(['ESTACION','anio']).ifs_disponible.mean().unstack()*100
plt.figure(figsize=(11,4));sns.heatmap(disponibilidad,annot=True,fmt='.0f',cmap='Blues',vmin=0,vmax=100,cbar_kws={'label':'IFS disponible (%)'})
plt.title('Figura 12. Disponibilidad de IFS: ausencia estructural en 2015–2016');plt.xlabel('Año');plt.ylabel('Estación')
fig_h2(12,'cobertura_ifs')

def n_series(col):
    values=[]
    for _,g in comp.sort_values('FECHA_HORA_LIMA').groupby('ESTACION'):
        values.append(hashlib.sha256(g[col].to_numpy(dtype=float).tobytes()).hexdigest())
    return len(set(values))
series_h2=tabla_h2([{'Variable':v,'Series ERA5 distintas':n_series('era5_'+v),'Series IFS distintas':n_series('ifs_'+v)}
                   for v in v_modelo],14,'Series meteorológicas distintas entre siete ubicaciones')
display(Markdown('ERA5 tiene una sola serie de viento para las siete ubicaciones; IFS tiene siete. No son siete observaciones espaciales independientes de ERA5. La dirección es **de procedencia** del viento, no una trayectoria de contaminación.'))
fig,axes=plt.subplots(2,4,subplot_kw={'projection':'polar'},figsize=(14,7))
for ax,(station,g) in zip(axes.flat,comp.groupby('ESTACION')):
    angles=g.loc[g.ifs_wind_speed_100m.gt(.1),'ifs_wind_direction_100m'].dropna().to_numpy()
    counts,edges=np.histogram(angles,bins=np.linspace(0,360,17))
    ax.bar(np.deg2rad((edges[:-1]+edges[1:])/2),100*counts/counts.sum(),width=np.deg2rad(21),color='#3182bd',alpha=.8)
    ax.set_theta_zero_location('N');ax.set_theta_direction(-1)
    ax.set_title(station.replace('_',' '),fontsize=9,pad=12)
axes.flat[-1].set_visible(False)
fig.suptitle('Figura 13. Procedencia del viento IFS a 100 m (porcentaje de horas; 2017–2024)',y=1.02)
fig_h2(13,'direccion_viento_ifs')
veh_cols=['hogares_con_auto_camioneta_2017','porcentaje_hogares_con_auto_camioneta_2017',
          'hogares_con_motocicleta_2017','porcentaje_hogares_con_motocicleta_2017']
vehiculos_h2=tabla_h2(vehiculos[['ESTACION']+veh_cols].sort_values(veh_cols[1],ascending=False),15,'Tenencia vehicular de los siete distritos, Censo 2017')
v=vehiculos_h2.sort_values(veh_cols[1])
plt.figure(figsize=(10,4.5));plt.barh(v.ESTACION.str.replace('_',' '),v[veh_cols[1]],color='#4169a1')
for i,value in enumerate(v[veh_cols[1]]):plt.text(value+.5,i,f'{value:.2f} %',va='center')
plt.xlim(0,75);plt.xlabel('Hogares con automóvil o camioneta (%)');plt.title('Figura 14. Tenencia de automóvil/camioneta en 2017')
fig_h2(14,'tenencia_vehicular')
tabla_calor=tabla_h2(calor.assign(anio=calor.FECHA_HORA_UTC.dt.year,
                                 vegetacion=calor.type.eq(0),alta_nominal=calor.confidence.isin(['n','h']))
    .groupby('anio',as_index=False).agg(detecciones=('type','size'),presunta_vegetacion=('vegetacion','sum'),
                                      nominal_o_alta=('alta_nominal','sum')),16,'Detecciones térmicas en Lima y su entorno por año UTC')
tipos_calor=calor.type.value_counts().sort_index()
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(tabla_calor.anio,tabla_calor.detecciones,color='#bd572e');axes[0].set(xlabel='Año UTC',ylabel='Detecciones',title='Registro anual (no incendios independientes)')
labels={0:'Presunta vegetación',1:'Volcán',2:'Fuente terrestre fija',3:'Sobre agua'}
axes[1].bar([labels[x] for x in tipos_calor.index],tipos_calor.values,color=['#50976c','#a98068','#6290b6'][:len(tipos_calor)])
axes[1].tick_params(axis='x',rotation=15);axes[1].set(ylabel='Detecciones',title='Tipo inferido por el producto')
fig.suptitle('Figura 15. Anomalías térmicas VIIRS S-NPP, 2015–2024');fig_h2(15,'detecciones_calor')
display(Markdown('La región se definió con latitud [−13,−11] y longitud [−78,−76], suficiente para los radios de 25/50 km usados. Una detección es un píxel, puede repetirse en otro paso del satélite y puede corresponder a una fuente fija. Cero detecciones no demuestra ausencia de fuego.'))

# EDA de asociaciones contemporáneas, no selección usando el objetivo futuro.
eda_nueva=aire.select(['ESTACION','FECHA_HORA_LIMA','PM2_5']).collect().to_pandas().merge(comp,on=['ESTACION','FECHA_HORA_LIMA'],how='left',validate='one_to_one')
candidatas_eda=['era5_wind_speed_100m','ifs_wind_speed_100m','era5_shortwave_radiation',
                'ifs_shortwave_radiation','ifs_cloud_cover_low','era5_dew_point_2m',
                'firms_detecciones_vegetacion_50km_24h','firms_frp_suma_50km_24h']
correlaciones_h2=tabla_h2([{'Variable':c,'Pares':int(eda_nueva[['PM2_5',c]].dropna().shape[0]),
                           'Pearson con PM2.5 actual':eda_nueva[['PM2_5',c]].corr().iloc[0,1]}
                          for c in candidatas_eda],17,'Relaciones exploratorias de los nuevos predictores con PM2.5 actual')
r=correlaciones_h2.sort_values('Pearson con PM2.5 actual')
plt.figure(figsize=(10,4));plt.barh(r.Variable,r['Pearson con PM2.5 actual'],color='#568d99');plt.axvline(0,color='gray',lw=1)
plt.xlabel('Correlación de Pearson');plt.title('Figura 16. Asociaciones contemporáneas; no son relaciones causales');fig_h2(16,'correlaciones_complementos')
RESUMEN_H2['complementos']={'filas':len(comp),'ifs_faltantes':int(comp.ifs_wind_speed_100m.isna().sum()),
                           'detecciones':len(calor),'tipos_calor':{str(k):int(v) for k,v in tipos_calor.items()},
                           'series_viento_era5':n_series('era5_wind_speed_100m'),'series_viento_ifs':n_series('ifs_wind_speed_100m')}
del eda_nueva


## 3. Data Preparation & Feature Engineering

### 3.1. Política de limpieza y unidad temporal


In [ ]:
import pandas as pd

columnas_necesarias = [
    "ESTACION", "FECHA_HORA_LIMA", "PM2_5", "temperature_2m",
    "relative_humidity_2m", "precipitation", "pressure_msl",
    "wind_speed_10m", "wind_direction_10m",
]
serie_pd = (
    unido.select(columnas_necesarias)
    .collect(engine="in-memory")
    .to_pandas()
    .sort_values(["ESTACION", "FECHA_HORA_LIMA"], kind="stable")
    .reset_index(drop=True)
)
serie_pd["FECHA_HORA_LIMA"] = pd.to_datetime(serie_pd["FECHA_HORA_LIMA"])
grupo = serie_pd.groupby("ESTACION", sort=False)
t = serie_pd["FECHA_HORA_LIMA"]
pm = serie_pd["PM2_5"]

# Verificamos horas reales, además del número de filas desplazadas.
antes_1h = grupo["FECHA_HORA_LIMA"].shift(1)
antes_5h = grupo["FECHA_HORA_LIMA"].shift(5)
antes_23h = grupo["FECHA_HORA_LIMA"].shift(23)
antes_24h = grupo["FECHA_HORA_LIMA"].shift(24)
despues_1h = grupo["FECHA_HORA_LIMA"].shift(-1)
serie_pd["pm25_lag_1h"] = grupo["PM2_5"].shift(1).where(
    (t - antes_1h) == pd.Timedelta(hours=1)
)
serie_pd["pm25_lag_24h"] = grupo["PM2_5"].shift(24).where(
    (t - antes_24h) == pd.Timedelta(hours=24)
)
serie_pd["pm25_media_6h"] = grupo["PM2_5"].transform(
    lambda x: x.rolling(6, min_periods=6).mean()
).where((t - antes_5h) == pd.Timedelta(hours=5))
serie_pd["objetivo_pm25_t_mas_1h"] = grupo["PM2_5"].shift(-1).where(
    (despues_1h - t) == pd.Timedelta(hours=1)
)

serie_pd["anio"] = t.dt.year
serie_pd["mes"] = t.dt.month
serie_pd["hora_lima"] = t.dt.hour
serie_pd["dia_semana"] = t.dt.dayofweek
serie_pd["fin_de_semana"] = serie_pd["dia_semana"] >= 5
serie_pd["hora_seno"] = np.sin(2 * np.pi * serie_pd["hora_lima"] / 24)
serie_pd["hora_coseno"] = np.cos(2 * np.pi * serie_pd["hora_lima"] / 24)
serie_pd["mes_seno"] = np.sin(2 * np.pi * serie_pd["mes"] / 12)
serie_pd["mes_coseno"] = np.cos(2 * np.pi * serie_pd["mes"] / 12)
serie_pd["viento_seno"] = np.sin(np.deg2rad(serie_pd["wind_direction_10m"]))
serie_pd["viento_coseno"] = np.cos(np.deg2rad(serie_pd["wind_direction_10m"]))
serie_pd["lluvia_en_t"] = (serie_pd["precipitation"] > 0).where(
    serie_pd["precipitation"].notna()
)


serie_pd["pm25_media_24h"] = grupo["PM2_5"].transform(
    lambda x: x.rolling(24, min_periods=24).mean()
).where((t - antes_23h) == pd.Timedelta(hours=23))
media24 = serie_pd["pm25_media_24h"].round(1)
categoria = np.select(
    [media24 <= 12.5, media24 <= 25, media24 <= 125],
    ["Buena", "Moderada", "Mala"],
    default="Umbral de cuidado"
)
serie_pd["categoria_inca_24h"] = pd.Series(
    categoria, dtype="string"
).where(media24.notna())
serie_pd["objetivo_categoria_inca_24h_t_mas_1h"] = (
    serie_pd.groupby("ESTACION", sort=False)["categoria_inca_24h"]
    .shift(-1)
    .where((despues_1h - t) == pd.Timedelta(hours=1))
)

total_filas = len(serie_pd)
pares_validos = serie_pd["PM2_5"].notna() & serie_pd["objetivo_pm25_t_mas_1h"].notna()
campos_requeridos = [
    "PM2_5", "objetivo_pm25_t_mas_1h", "pm25_lag_1h", "pm25_lag_24h",
    "pm25_media_6h", "temperature_2m", "relative_humidity_2m",
    "precipitation", "pressure_msl", "wind_speed_10m",
    "viento_seno", "viento_coseno",
]
completo = serie_pd[campos_requeridos].notna().all(axis=1)

serie_pd["periodo_mes"] = t.dt.to_period("M")
cobertura_mes = (
    serie_pd.groupby(["ESTACION", "periodo_mes"], observed=True)["PM2_5"]
    .agg(["size", "count"])
    .assign(cobertura=lambda x: x["count"] / x["size"])["cobertura"]
)
serie_pd = serie_pd.join(cobertura_mes, on=["ESTACION", "periodo_mes"])
modelo_base_pd = serie_pd.loc[completo].copy()
modelo_cobertura75_pd = modelo_base_pd.loc[modelo_base_pd["cobertura"] >= 0.75].copy()
modelo_clasificacion_pd = modelo_base_pd.dropna(subset=[
    "categoria_inca_24h",
    "objetivo_categoria_inca_24h_t_mas_1h",
]).copy()

resumen_features = pl.DataFrame([{
    "filas_originales": total_filas,
    "pm25_actual_disponible": int(pm.notna().sum()),
    "pares_t_y_t_mas_1h": int(pares_validos.sum()),
    "filas_completas_predictoras": len(modelo_base_pd),
    "filas_cobertura_mes_75": len(modelo_cobertura75_pd),
    "filas_clasificacion_completas": len(modelo_clasificacion_pd),
    "medias_24h_completas": int(media24.notna().sum()),
    "pares_para_clasificacion_24h": int((
        serie_pd["categoria_inca_24h"].notna() &
        serie_pd["objetivo_categoria_inca_24h_t_mas_1h"].notna()
    ).sum()),
}])
mostrar(resumen_features, "Resumen de preparación base 28 — Efecto real de eliminar filas sin predictores/objetivo")

resumen_por_anio_pd = (
    serie_pd.assign(pares_validos=pares_validos, completo=completo,
                    completo_75=completo & (serie_pd["cobertura"] >= 0.75))
    .groupby("anio", as_index=False)
    .agg(filas=("PM2_5", "size"), validas_pm25=("PM2_5", "count"),
         pares=("pares_validos", "sum"),
         filas_modelo=("completo", "sum"),
         filas_modelo_cobertura75=("completo_75", "sum"))
)
cobertura_features_anual = pl.from_pandas(resumen_por_anio_pd)
mostrar(cobertura_features_anual,
        "Resumen de preparación base 28A — Filas conservadas por año tras seleccionar observaciones útiles")
resumen_estacion_pd = (
    serie_pd.assign(completo=completo,
                    completo_75=completo & (serie_pd["cobertura"] >= 0.75))
    .groupby("ESTACION", as_index=False, observed=True)
    .agg(pm25_mediciones=("PM2_5", "count"),
         pm25_media_original=("PM2_5", "mean"),
         filas_modelo=("completo", "sum"),
         filas_modelo_cobertura75=("completo_75", "sum"))
)
resumen_estacion_pd["pm25_media_modelo"] = resumen_estacion_pd["ESTACION"].map(
    modelo_base_pd.groupby("ESTACION", observed=True)["PM2_5"].mean()
)
resumen_estacion_pd["pm25_media_cobertura75"] = resumen_estacion_pd["ESTACION"].map(
    modelo_cobertura75_pd.groupby("ESTACION", observed=True)["PM2_5"].mean()
)
retencion_por_estacion = pl.from_pandas(resumen_estacion_pd)
mostrar(retencion_por_estacion,
        "Resumen de preparación base 28B — Retención por estación y posible cambio en media de PM2.5")
print("Mediciones válidas de 2019-2021 que se perderían al borrar los años:",
      int(serie_pd.loc[serie_pd["anio"].between(2019, 2021), "PM2_5"].count()))

conteo_categorias = pl.from_pandas(
    serie_pd["categoria_inca_24h"].value_counts(dropna=False)
    .rename_axis("categoria_inca_24h")
    .reset_index(name="horas")
)
mostrar(conteo_categorias,
        "Resumen de preparación base 29 — Categorías INCA provisionales; solo promedios completos de 24 h")
print("PM2_5 se expresa en µg/m³ según el cruce con OpenAQ.")

categorias_graf = conteo_categorias.to_pandas().dropna(
    subset=["categoria_inca_24h"]
)
plt.figure(figsize=(8, 4))
color_inca = {"Buena": "#519b5e", "Moderada": "#e6c153",
              "Mala": "#e6954c", "Umbral de cuidado": "#c96258"}
plt.bar(categorias_graf["categoria_inca_24h"],
        categorias_graf["horas"],
        color=[color_inca[c] for c in categorias_graf["categoria_inca_24h"]])
plt.ylabel("Horas con promedio móvil completo de 24 h")
plt.title("Categoría INCA provisional de PM2.5 (supone µg/m³)")
figura("fig09_categoria_inca_24h_provisional.png")

columnas_modelo = [
    "ESTACION", "FECHA_HORA_LIMA", "PM2_5", "objetivo_pm25_t_mas_1h",
    "pm25_lag_1h", "pm25_lag_24h", "pm25_media_6h",
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "pressure_msl", "wind_speed_10m", "viento_seno", "viento_coseno",
    "hora_seno", "hora_coseno", "mes_seno", "mes_coseno",
    "fin_de_semana", "lluvia_en_t",
    "pm25_media_24h", "categoria_inca_24h",
    "objetivo_categoria_inca_24h_t_mas_1h",
]
RUTA_FEATURES = SALIDA / "modelo_pm25_solo_columnas_utiles.parquet"
RUTA_COBERTURA75 = SALIDA / "modelo_pm25_cobertura_mensual_75.parquet"
RUTA_CLASIFICACION = SALIDA / "modelo_clasificacion_inca_24h.parquet"
archivo_anterior = SALIDA / "variables_candidatas_pm25.parquet"
if archivo_anterior.exists():
    archivo_anterior.unlink()
modelo_base_pd[columnas_modelo].to_parquet(RUTA_FEATURES, index=False)
modelo_cobertura75_pd[columnas_modelo].to_parquet(RUTA_COBERTURA75, index=False)
modelo_clasificacion_pd[columnas_modelo].to_parquet(RUTA_CLASIFICACION, index=False)
print("Columnas exportadas:", columnas_modelo)
print("Conjunto principal:", RUTA_FEATURES.name, "filas:", len(modelo_base_pd))
print("Prueba de cobertura >=75%:", RUTA_COBERTURA75.name,
      "filas:", len(modelo_cobertura75_pd))
print("Conjunto con categorías de t y t+1 completas:",
      RUTA_CLASIFICACION.name, "filas:", len(modelo_clasificacion_pd))
print("Capa límite no es entrada: faltaría en todas las horas de 2024.")
print("Para operar a futuro, sustituir ERA5 histórico por clima disponible al emitir predicción.")

diccionario_features = pl.DataFrame([
    {"columnas": "ESTACION, FECHA_HORA_LIMA", "uso": "Identidad y partición temporal"},
    {"columnas": "PM2_5, lag_1h, lag_24h, media_6h", "uso": "Concentración actual e historial disponible en t"},
    {"columnas": "temperatura, humedad, lluvia, presión, viento", "uso": "Meteorología de t; validar disponibilidad operativa"},
    {"columnas": "hora/mes cíclicos, fin_de_semana", "uso": "Calendario sin datos futuros"},
    {"columnas": "categoria_inca_24h", "uso": "Descripción en t; indicador exploratorio de 24 h"},
    {"columnas": "objetivo_pm25_t_mas_1h", "uso": "Variable objetivo de regresión; NO predictor"},
    {"columnas": "objetivo_categoria_inca_24h_t_mas_1h", "uso": "Objetivo alternativo de clasificación; NO predictor"},
])
mostrar(diccionario_features, "Resumen de preparación base 30 — Columnas finales y roles")


### 3.2. Integración, ingeniería y partición


In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CLAVE=['ESTACION','FECHA_HORA_LIMA']
OBJETIVO='objetivo_pm25_t_mas_1h'
base_modelo=modelo_base_pd.copy()
# Variables construidas mientras permanecen todas las horas, incluidas las faltantes.
serie_pd=serie_pd.sort_values(CLAVE).reset_index(drop=True)
g=serie_pd.groupby('ESTACION',sort=False)
delta=serie_pd.FECHA_HORA_LIMA-g.FECHA_HORA_LIMA.shift(23)
serie_pd['pm25_estacional_24h']=g.PM2_5.shift(23).where(delta.eq(pd.Timedelta(hours=23)))
serie_pd['pm25_lag_3h']=g.PM2_5.shift(3).where((serie_pd.FECHA_HORA_LIMA-g.FECHA_HORA_LIMA.shift(3)).eq(pd.Timedelta(hours=3)))
serie_pd['pm25_std_6h']=g.PM2_5.transform(lambda s:s.rolling(6,min_periods=6).std())
serie_pd['pm25_amplitud_6h']=g.PM2_5.transform(lambda s:s.rolling(6,min_periods=6).max()-s.rolling(6,min_periods=6).min())
ingenieria_pm=['pm25_lag_3h','pm25_std_6h','pm25_amplitud_6h']
base_modelo=base_modelo.merge(serie_pd[CLAVE+ingenieria_pm+['pm25_estacional_24h']],on=CLAVE,how='left',validate='one_to_one')
base_modelo['pm25_cambio_1h']=base_modelo.PM2_5-base_modelo.pm25_lag_1h
base_modelo['pm25_cambio_24h']=base_modelo.PM2_5-base_modelo.pm25_lag_24h

columnas_extra=[c for c in comp if c.startswith(('era5_','ifs_'))]
datos_nuevos=comp[CLAVE+columnas_extra].copy()
columnas_firms=[c for c in comp if c.startswith('firms_')]
firm_retardo=comp[CLAVE+columnas_firms].copy()
# Mover la clave hacia delante equivale a usar la ventana del archivo a t-3 h.
firm_retardo['FECHA_HORA_LIMA']+=pd.Timedelta(hours=3)
firm_retardo=firm_retardo.rename(columns={c:c+'_retardo3h' for c in columnas_firms})
datos_nuevos=datos_nuevos.merge(firm_retardo,on=CLAVE,how='left',validate='one_to_one')
modelo=base_modelo.merge(datos_nuevos,on=CLAVE,how='left',validate='one_to_one',indicator='_union_comp')
assert modelo._union_comp.eq('both').all()
censo_h2=pd.read_csv(RUTA_DEMO_CENSO,dtype={'UBIGEO':str})
censo_h2['censo_poblacion_2017']=censo_h2.POBLACION_CENSADA_2017
censo_h2['censo_pct_0_4_2017']=100*censo_h2.EDAD_0_4_2017/censo_h2.POBLACION_CENSADA_2017
censo_h2['censo_pct_65_mas_2017']=100*censo_h2.EDAD_65_MAS_2017/censo_h2.POBLACION_CENSADA_2017
contexto_cols=['censo_poblacion_2017','censo_pct_0_4_2017','censo_pct_65_mas_2017']+veh_cols
modelo=modelo.merge(censo_h2[['ESTACION']+contexto_cols[:3]],on='ESTACION',how='left',validate='many_to_one')
modelo=modelo.merge(vehiculos[['ESTACION']+veh_cols],on='ESTACION',how='left',validate='many_to_one')
assert modelo[contexto_cols].notna().all().all()
modelo['FECHA_OBJETIVO']=modelo.FECHA_HORA_LIMA+pd.Timedelta(hours=1)
for radio in [25,50]:
    c=f'firms_frp_suma_{radio}km_24h_retardo3h'
    modelo[f'firms_log_frp_{radio}km_24h_retardo3h']=np.log1p(modelo[c])
modelo=modelo.sort_values(['FECHA_HORA_LIMA','ESTACION']).reset_index(drop=True)

uniones_h2=tabla_h2([
 {'Cruce':'Aire completo → clima original','Filas de origen':len(serie_pd),'Sin coincidencia':int(serie_pd.temperature_2m.isna().sum()),'Acción':'No desplazar reloj del aire; excluir si falta meteorología básica'},
 {'Cruce':'Filas utilizables → complementos','Filas de origen':len(modelo),'Sin coincidencia':int(modelo._union_comp.ne('both').sum()),'Acción':'Unión 1:1 por estación y hora'},
 {'Cruce':'Filas utilizables → Censo 2017 / vehículos','Filas de origen':len(modelo),'Sin coincidencia':int(modelo[contexto_cols].isna().any(axis=1).sum()),'Acción':'Unión muchos:1 por estación / distrito'}],18,'Control de uniones y efecto de la referencia horaria')

F_BASE=['PM2_5','pm25_lag_1h','pm25_lag_24h','pm25_media_6h','temperature_2m',
        'relative_humidity_2m','precipitation','pressure_msl','wind_speed_10m',
        'viento_seno','viento_coseno','hora_seno','hora_coseno','mes_seno','mes_coseno',
        'dia_semana','fin_de_semana','anio','lluvia_en_t','ESTACION']
F_HIST=ingenieria_pm+['pm25_cambio_1h','pm25_cambio_24h']
F_ERA=['era5_wind_speed_100m','era5_viento_u_100m','era5_viento_v_100m',
       'era5_shortwave_radiation','era5_cloud_cover','era5_cloud_cover_low',
       'era5_surface_pressure','era5_dew_point_2m']
F_IFS=['ifs_wind_speed_100m','ifs_viento_u_100m','ifs_viento_v_100m','ifs_shortwave_radiation',
       'ifs_cloud_cover','ifs_cloud_cover_low','ifs_surface_pressure','ifs_dew_point_2m','ifs_disponible']
F_FIRMS=['firms_detecciones_vegetacion_25km_24h_retardo3h','firms_detecciones_vegetacion_50km_24h_retardo3h',
         'firms_log_frp_25km_24h_retardo3h','firms_log_frp_50km_24h_retardo3h',
         'firms_ventana_archivo_disponible_24h_retardo3h']
F_CONTEXTO=contexto_cols
F_TODAS=F_BASE+F_HIST+F_ERA+F_IFS+F_FIRMS+F_CONTEXTO
assert len(F_TODAS)==len(set(F_TODAS))
assert not any('objetivo' in c.lower() or 'categoria_inca' in c.lower() for c in F_TODAS)
assert not set(['PM10','NO2','ID','LATITUD','LONGITUD','boundary_layer_height','FECHA_HORA_LIMA']).intersection(F_TODAS)
assert modelo[F_BASE+F_HIST+[OBJETIVO]].notna().all().all()
for c in F_TODAS:
    if c!='ESTACION': modelo[c]=pd.to_numeric(modelo[c],errors='raise').astype('float64')

roles_h2=tabla_h2([
 {'Grupo':'PM2.5 e historia','Cantidad':9,'Transformación':'lags, media/dispersion 6 h, cambios 1/24 h','Tratamiento':'Historia real, sin imputar observaciones'},
 {'Grupo':'Meteorología original y calendario','Cantidad':len(F_BASE)-4-1,'Transformación':'ángulos en seno/coseno; ciclos hora/mes','Tratamiento':'Completa en filas seleccionadas'},
 {'Grupo':'Meteorología complementaria ERA5','Cantidad':len(F_ERA),'Transformación':'viento u/v a 100 m y variables físicas','Tratamiento':'Mediana + indicador si faltara'},
 {'Grupo':'Meteorología complementaria IFS','Cantidad':len(F_IFS),'Transformación':'viento u/v a 100 m; fuente separada','Tratamiento':'Mediana e indicadores solo desde train; ausente 2015–2016'},
 {'Grupo':'FIRMS','Cantidad':len(F_FIRMS),'Transformación':'ventana pasada 24 h, retardo 3 h y log1p FRP','Tratamiento':'Ausencia de archivo separada de cero detecciones'},
 {'Grupo':'Censo/vehículos 2017','Cantidad':len(F_CONTEXTO),'Transformación':'referencia fija por distrito','Tratamiento':'No son una serie anual ni aforo'},
 {'Grupo':'Estación / objetivo','Cantidad':2,'Transformación':'One-Hot / PM2.5 t+1 h','Tratamiento':'Fecha ordena; objetivo siempre fuera de X'}],19,'Grupos de variables y decisiones de preparación')
flujo_h2=tabla_h2([
 {'Etapa':'Horas originales de aire','Filas':len(serie_pd)},
 {'Etapa':'Con PM2.5 actual observado','Filas':int(serie_pd.PM2_5.notna().sum())},
 {'Etapa':'Con predictores básicos y objetivo completos','Filas':len(base_modelo)},
 {'Etapa':'Tras unión y variables nuevas (antes de purga)','Filas':len(modelo)},
 {'Etapa':'Comparación: cobertura mensual 75 % (no adoptada)','Filas':len(modelo_cobertura75_pd)},
 {'Etapa':'Conjunto exploratorio de clasificación (no entrenado)','Filas':len(modelo_clasificacion_pd)}],20,'Balance de filas de la preparación recalculada')

CORTE_TRAIN=pd.Timestamp('2022-01-01')
CORTE_TEST=pd.Timestamp('2023-01-01')
# La pertenencia depende de la hora de la etiqueta. Una fila que cruza un corte se purga.
train=modelo.loc[modelo.FECHA_OBJETIVO.lt(CORTE_TRAIN)].copy()
val=modelo.loc[modelo.FECHA_HORA_LIMA.ge(CORTE_TRAIN)&modelo.FECHA_OBJETIVO.lt(CORTE_TEST)].copy()
test=modelo.loc[modelo.FECHA_HORA_LIMA.ge(CORTE_TEST)].copy()
dev=modelo.loc[modelo.FECHA_OBJETIVO.lt(CORTE_TEST)].copy()
assert train.FECHA_OBJETIVO.max()<val.FECHA_HORA_LIMA.min()
assert dev.FECHA_OBJETIVO.max()<test.FECHA_HORA_LIMA.min()
assert set(train.index).isdisjoint(val.index) and set(dev.index).isdisjoint(test.index)
purga=len(modelo)-len(train)-len(val)-len(test)
particiones_h2=tabla_h2([{'Partición':name,'Filas':len(df),'Estaciones':df.ESTACION.nunique(),
                         'Primera hora objetivo':str(df.FECHA_OBJETIVO.min()),
                         'Última hora objetivo':str(df.FECHA_OBJETIVO.max())}
                        for name,df in [('Entrenamiento',train),('Validación',val),('Prueba',test)]]
                       +[{'Partición':'Purga de límites (t+1 cruza corte)','Filas':purga}],21,'Partición cronológica y purga de objetivos en los límites')
conteos_estacion=pd.DataFrame({'Train':train.groupby('ESTACION').size(),'Validación':val.groupby('ESTACION').size(),
                              'Prueba':test.groupby('ESTACION').size()}).fillna(0).astype(int)
display(conteos_estacion)
conteos_estacion.to_csv(SALIDA/'soporte_estacion_particion.csv',encoding='utf-8-sig')
fig,axes=plt.subplots(1,2,figsize=(13,4))
conteos_estacion.plot.barh(ax=axes[0],color=['#506f8e','#88a96a','#cd9862']);axes[0].set_xlabel('Filas utilizables')
daily=modelo.groupby(modelo.FECHA_HORA_LIMA.dt.floor('D')).size()
axes[1].plot(daily.index,daily.values,color='#536f89',lw=.7)
axes[1].axvline(CORTE_TRAIN,color='#77a252',label='Comienza validación');axes[1].axvline(CORTE_TEST,color='#cd8f4a',label='Comienza prueba')
axes[1].set(ylabel='Filas utilizables por día',xlabel='Fecha');axes[1].legend(fontsize=8)
#Figura 17
fig.suptitle('Disponibilidad real y partición temporal');fig_h2(17,'particiones')

def preprocesador(columnas):
    numericas=[c for c in columnas if c!='ESTACION']
    return ColumnTransformer([
      ('numericas',Pipeline([('imputar',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),
                            ('escalar',StandardScaler())]),numericas),
      ('estacion',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['ESTACION'])],remainder='drop')

def crear_pipeline(estimator,columnas):
    return Pipeline([('preparar',preprocesador(columnas)),('modelo',estimator)])

diccionario_final=pd.DataFrame([{'Columna':c,'Rol':'predictor','Grupo':'categórica' if c=='ESTACION' else 'numérica'} for c in F_TODAS]
   +[{'Columna':OBJETIVO,'Rol':'objetivo, nunca predictor','Grupo':'concentración'},
     {'Columna':'FECHA_HORA_LIMA / FECHA_OBJETIVO','Rol':'orden, unión y partición','Grupo':'fecha/hora Perú'}])
diccionario_final.to_csv(SALIDA/'diccionario_modelo.csv',index=False,encoding='utf-8-sig')
modelo[CLAVE+['FECHA_OBJETIVO',OBJETIVO,'pm25_estacional_24h']+[c for c in F_TODAS if c!='ESTACION']].to_parquet(SALIDA/'dataset_modelado_hito2.parquet',index=False)
RESUMEN_H2['preparacion']={'filas_antes_purga':len(modelo),'predictores':len(F_TODAS),
    'train':len(train),'val':len(val),'test':len(test),'purga':purga,
    'cobertura75':len(modelo_cobertura75_pd),'clasificacion':len(modelo_clasificacion_pd),
    'hora_senamhi':HORA_SENAMHI,'proyecciones_filas':len(pd.read_csv(RUTA_DEMO_PROY)),
    'proyecciones_2018_2024':int(pd.read_csv(RUTA_DEMO_PROY).ANIO.between(2018,2024).sum())}
print('Filas preparadas:',len(modelo),'Predictores:',len(F_TODAS),'Purga:',purga)


## 4. Modeling — modelos y desempeño inicial

### 4.1. Diseño del experimento
Todas las comparaciones tabulares usan las mismas filas de entrenamiento y validación. Se comparan regresión lineal, Ridge, Elastic Net, Huber, Linear SVR, árbol, Bagging, Random Forest, Extra Trees, AdaBoost, Gradient Boosting, HistGradientBoosting, XGBoost, LightGBM, CatBoost, KNN y MLP.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet, HuberRegressor
from sklearn.svm import LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (BaggingRegressor,RandomForestRegressor,ExtraTreesRegressor,
                             AdaBoostRegressor,GradientBoostingRegressor,HistGradientBoostingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

def evaluar(df,pred):
    y=df[OBJETIVO].to_numpy(dtype=float)
    pred=np.maximum(np.asarray(pred,dtype=float),0)
    if len(pred)!=len(y) or not np.isfinite(pred).all(): raise ValueError('Predicciones incompletas/no finitas')
    errores=pd.DataFrame({'ESTACION':df.ESTACION.to_numpy(),'abs':np.abs(y-pred)})
    return {'MAE_global':mean_absolute_error(y,pred),'MAE_macro_estacion':errores.groupby('ESTACION')['abs'].mean().mean(),
            'RMSE':np.sqrt(mean_squared_error(y,pred)),'R2':r2_score(y,pred),'N_validacion':len(y)}

def predecir_en_lotes(pipe,X,batch=4096):
    return np.maximum(np.concatenate([pipe.predict(X.iloc[i:i+batch]) for i in range(0,len(X),batch)]),0)

specs={
 'Lineal':LinearRegression(),
 'Ridge':Ridge(alpha=10),
 'ElasticNet':ElasticNet(alpha=.03,l1_ratio=.2,max_iter=1500,tol=.001,random_state=SEED),
 'Huber':HuberRegressor(epsilon=1.35,alpha=.01,max_iter=150,tol=.0001),
 'LinearSVR':LinearSVR(C=1,epsilon=.1,dual='auto',max_iter=3000,random_state=SEED),
 'Arbol':DecisionTreeRegressor(max_depth=10,min_samples_leaf=30,random_state=SEED),
 'Bagging':BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=12,min_samples_leaf=30),
                           n_estimators=20,max_samples=.5,max_features=.8,n_jobs=1,random_state=SEED),
 'RandomForest':RandomForestRegressor(n_estimators=60,max_depth=16,min_samples_leaf=25,
                                     max_features=.7,max_samples=.6,n_jobs=N_JOBS,random_state=SEED),
 'ExtraTrees':ExtraTreesRegressor(n_estimators=60,max_depth=16,min_samples_leaf=25,max_features=.7,n_jobs=N_JOBS,random_state=SEED),
 'AdaBoost':AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=3,min_samples_leaf=30),
                            n_estimators=50,learning_rate=.05,random_state=SEED,loss='linear'),
 'GradientBoosting':GradientBoostingRegressor(n_estimators=70,max_depth=3,min_samples_leaf=30,learning_rate=.05,random_state=SEED),
 'HistGB':HistGradientBoostingRegressor(max_iter=120,max_leaf_nodes=31,min_samples_leaf=40,
                                      learning_rate=.08,l2_regularization=1,early_stopping=False,random_state=SEED),
 'XGBoost':XGBRegressor(n_estimators=220,max_depth=5,learning_rate=.05,subsample=.8,
                      colsample_bytree=.8,reg_lambda=5,tree_method='hist',n_jobs=N_JOBS,random_state=SEED),
 'LightGBM':LGBMRegressor(n_estimators=220,num_leaves=31,max_depth=-1,min_child_samples=40,
                        learning_rate=.05,reg_lambda=5,verbosity=-1,n_jobs=N_JOBS,random_state=SEED),
 'CatBoost':CatBoostRegressor(iterations=220,depth=6,learning_rate=.05,loss_function='RMSE',
                            l2_leaf_reg=5,random_seed=SEED,thread_count=N_JOBS,verbose=False,allow_writing_files=False),
 'KNN':KNeighborsRegressor(n_neighbors=15,weights='distance',algorithm='brute',n_jobs=N_JOBS),
 'MLP':TransformedTargetRegressor(regressor=MLPRegressor(hidden_layer_sizes=(32,16),batch_size=2048,
       max_iter=100,learning_rate_init=.003,early_stopping=False,tol=.0005,random_state=SEED),transformer=StandardScaler())}

hiperparametros_h2=tabla_h2([{'Modelo':name,'Configuración':str(est.get_params(deep=False)),
                             'Entrenamiento':len(train),'Predictores':len(F_TODAS)} for name,est in specs.items()],22,'Modelos y configuraciones fijadas antes de validar',limite=25)
resultados_val=[];modelos_entrenados={};features_modelo={};pred_val={};errores_modelo=[]
for name,pred in [('Persistencia',val.PM2_5.to_numpy()),
                 ('Estacional24h',val.pm25_estacional_24h.fillna(val.PM2_5).to_numpy()),
                 ('Media_train',np.repeat(train[OBJETIVO].mean(),len(val)))]:
    resultados_val.append({'Modelo':name,**evaluar(val,pred),'Fit_s':0.,'Predict_s':0.,
                           'MAE_train_muestra':np.nan,'Advertencia':'','Elegible':False})
    pred_val[name]=np.maximum(pred,0)

train_auditoria=train.sample(n=min(3000,len(train)),random_state=SEED)
def entrenar_validar(nombre,estimador,features):
    print('ENTRENANDO',nombre,'filas',len(train),'features',len(features),flush=True)
    pipe=crear_pipeline(clone(estimador),features)
    start=time.perf_counter()
    with warnings.catch_warnings(record=True) as ws:
        warnings.simplefilter('always')
        pipe.fit(train[features],train[OBJETIVO])
    fit_s=time.perf_counter()-start
    start=time.perf_counter();pred=predecir_en_lotes(pipe,val[features]);pred_s=time.perf_counter()-start
    pred_train=predecir_en_lotes(pipe,train_auditoria[features])
    notas=' | '.join(sorted({str(w.message)[:180] for w in ws}))
    registro={'Modelo':nombre,**evaluar(val,pred),'Fit_s':fit_s,'Predict_s':pred_s,
              'MAE_train_muestra':mean_absolute_error(train_auditoria[OBJETIVO],pred_train),
              'Advertencia':notas,'Elegible':True}
    resultados_val.append(registro);modelos_entrenados[nombre]=pipe;features_modelo[nombre]=features.copy();pred_val[nombre]=pred
    pd.DataFrame(resultados_val).to_csv(SALIDA/'comparacion_validacion_en_progreso.csv',index=False)
    print('VALIDADO',nombre,'MAE_macro',round(registro['MAE_macro_estacion'],4),'seg',round(fit_s+pred_s,1),flush=True)
    return registro

for nombre,estimador in specs.items():
    try: entrenar_validar(nombre,estimador,F_TODAS)
    except Exception as exc:
        errores_modelo.append({'Modelo':nombre,'Error':repr(exc)})
        print('FALLO_REGISTRADO',nombre,repr(exc),flush=True)
if errores_modelo:
    display(pd.DataFrame(errores_modelo))
    pd.DataFrame(errores_modelo).to_csv(SALIDA/'modelos_no_completados.csv',index=False)
display(pd.DataFrame(resultados_val).sort_values('MAE_macro_estacion')[['Modelo','MAE_global','MAE_macro_estacion','RMSE','R2','Fit_s','Predict_s']])
print('Prueba temporal todavía no utilizada para comparar modelos.')


### 4.2. ARIMA por estación y comparación de complementos
ARIMA se ajusta por estación sobre una ventana de 180 días de calendario que termina en la última medición anterior al corte, ampliada con las horas faltantes hasta el límite temporal. Se mantiene frecuencia horaria y se dejan ausencias como NaN en el modelo de espacio de estados. Se prueban órdenes (1,0,0), (2,0,0) y (1,1,1). No se concatenan fragmentos separados por meses.


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

series_pm={s:g.set_index('FECHA_HORA_LIMA').PM2_5.sort_index().asfreq('h') for s,g in serie_pd.groupby('ESTACION')}
registros_arima=[]
ARIMA_ORDENES={'ARIMA_100':(1,0,0),'ARIMA_200':(2,0,0),'ARIMA_111':(1,1,1)}

def predecir_arima(order,frame,fin_ajuste,etapa):
    result=pd.Series(index=frame.index,dtype=float)
    completos=True
    for station,evaluables in frame.groupby('ESTACION'):
        raw_series=series_pm[station]
        disponibles=raw_series.loc[:fin_ajuste].dropna()
        if len(disponibles)<168:
            raise ValueError('Historia ARIMA insuficiente para '+station)
        last_valid=disponibles.index[-1]
        start=max(raw_series.index.min(),last_valid-pd.Timedelta(days=180))
        serie_ajuste=raw_series.loc[start:fin_ajuste].asfreq('h')
        kwargs={'order':order,'trend':'c' if order[1]==0 else 'n',
                'enforce_stationarity':False,'enforce_invertibility':False}
        print('ARIMA',etapa,order,station,'observados',serie_ajuste.notna().sum(),'horas',len(serie_ajuste),flush=True)
        t0=time.perf_counter()
        with warnings.catch_warnings(record=True) as ws:
            warnings.simplefilter('always')
            fitted=SARIMAX(serie_ajuste,**kwargs).fit(disp=False,maxiter=100)
        converged=bool(fitted.mle_retvals.get('converged',False))
        # Filtrar (no suavizar) toda la secuencia con parámetros entrenados.
        # predicted_mean[u] solo condiciona en las observaciones hasta u-1.
        secuencia=raw_series.loc[start:evaluables.FECHA_OBJETIVO.max()].asfreq('h')
        filtered=SARIMAX(secuencia,**kwargs).filter(fitted.params)
        predicted=filtered.get_prediction(start=0,end=len(secuencia)-1,dynamic=False).predicted_mean
        p=predicted.reindex(pd.DatetimeIndex(evaluables.FECHA_OBJETIVO)).to_numpy()
        if not np.isfinite(p).all(): completos=False
        result.loc[evaluables.index]=np.maximum(p,0)
        registros_arima.append({'Etapa':etapa,'Orden':str(order),'ESTACION':station,
            'Inicio ajuste':str(start),'Fin ajuste':str(fin_ajuste),'Horas':len(serie_ajuste),
            'Observadas':int(serie_ajuste.notna().sum()),'Faltantes':int(serie_ajuste.isna().sum()),
            'AIC':float(fitted.aic),'Convergió':converged,'Fit_s':time.perf_counter()-t0,
            'Advertencia':' | '.join(sorted({str(w.message)[:150] for w in ws})),
            'Parametros':json.dumps({str(k):float(v) for k,v in fitted.params.items()})})
        completos=completos and converged
        del filtered,fitted
    return result.to_numpy(),completos

for name,order in ARIMA_ORDENES.items():
    try:
        t0=time.perf_counter();p,ok=predecir_arima(order,val,CORTE_TRAIN-pd.Timedelta(hours=1),'validacion')
        registro={'Modelo':name,**evaluar(val,p),'Fit_s':time.perf_counter()-t0,'Predict_s':np.nan,
                  'MAE_train_muestra':np.nan,'Advertencia':'' if ok else 'Algún ajuste no convergió; no elegible','Elegible':ok}
        resultados_val.append(registro);pred_val[name]=p
        print('ARIMA_VALIDADO',name,round(registro['MAE_macro_estacion'],4),'elegible',ok,flush=True)
    except Exception as exc:
        errores_modelo.append({'Modelo':name,'Error':repr(exc)})
        print('ARIMA_FALLO',name,repr(exc),flush=True)
auditoria_arima_h2=tabla_h2(pd.DataFrame(registros_arima).drop(columns=['Parametros','Advertencia']),24,'ARIMA: historia horaria, faltantes y convergencia por estación')
pd.DataFrame(registros_arima).to_csv(SALIDA/'auditoria_arima_completa.csv',index=False,encoding='utf-8-sig')

ablaciones={
 'HGB_original':F_BASE,
 'HGB_historia':F_BASE+F_HIST,
 'HGB_ERA5':F_BASE+F_HIST+F_ERA,
 'HGB_IFS':F_BASE+F_HIST+F_ERA+F_IFS,
 'HGB_FIRMS':F_BASE+F_HIST+F_ERA+F_IFS+F_FIRMS,
 'HistGB':F_TODAS}
ablation_results=[]
for name,features in ablaciones.items():
    if name=='HistGB':
        r=next(r for r in resultados_val if r['Modelo']==name)
    else:r=entrenar_validar(name,specs['HistGB'],features)
    ablation_results.append({'Configuración':name,'Predictores':len(features),
                            'MAE_global':r['MAE_global'],'MAE_macro_estacion':r['MAE_macro_estacion'],'RMSE':r['RMSE']})
ablacion_h2=tabla_h2(ablation_results,25,'Aporte incremental con el mismo HistGradientBoosting y las mismas filas')
plt.figure(figsize=(10,4));plt.plot(ablacion_h2.Configuración,ablacion_h2.MAE_macro_estacion,marker='o',color='#577e8e')
plt.xticks(rotation=15);plt.ylabel('MAE medio entre estaciones');plt.title('Aporte incremental medido en validación');fig_h2(19,'ablacion')
comparacion_val_h2=pd.DataFrame(resultados_val).sort_values(['MAE_macro_estacion','MAE_global']).reset_index(drop=True)
comparacion_val_h2['Mejora_macro_vs_persistencia_pct']=100*(1-comparacion_val_h2.MAE_macro_estacion/
    next(r['MAE_macro_estacion'] for r in resultados_val if r['Modelo']=='Persistencia'))
tabla_h2(comparacion_val_h2[['Modelo','MAE_global','MAE_macro_estacion','RMSE','R2','Fit_s','Predict_s','Elegible']],23,'Comparación completa en validación (mismas horas objetivo)',limite=40)
comparacion_val_h2.to_csv(SALIDA/'comparacion_modelos_validacion.csv',index=False,encoding='utf-8-sig')
best=comparacion_val_h2.loc[comparacion_val_h2.Elegible].iloc[0]
MODELO_ELEGIDO=best.Modelo
MODELO_TABULAR_ELEGIDO=comparacion_val_h2.loc[comparacion_val_h2.Modelo.isin(modelos_entrenados)].iloc[0].Modelo
elegibles_ar=comparacion_val_h2.loc[comparacion_val_h2.Modelo.isin(ARIMA_ORDENES)&comparacion_val_h2.Elegible]
ARIMA_ELEGIDO=elegibles_ar.iloc[0].Modelo if len(elegibles_ar) else None
print('SELECCIÓN CERRADA ANTES DE PRUEBA:',MODELO_ELEGIDO,'MAE macro:',best.MAE_macro_estacion)
fig,ax=plt.subplots(figsize=(11,8))
v=comparacion_val_h2.sort_values('MAE_macro_estacion',ascending=False)
sin_convergencia=v.Modelo.isin(ARIMA_ORDENES)&~v.Elegible
colors=['#bb7e35' if m=='Persistencia' else '#aaaaaa' if sc else '#4f7897' for m,sc in zip(v.Modelo,sin_convergencia)]
labels=[m+' *' if sc else m for m,sc in zip(v.Modelo,sin_convergencia)]
ax.barh(labels,v.MAE_macro_estacion,color=colors)
ax.axvline(next(r['MAE_macro_estacion'] for r in resultados_val if r['Modelo']=='Persistencia'),color='#bb7e35',linestyle='--',label='Persistencia')
ax.set(xlabel='MAE medio entre estaciones',title='Comparación temporal en validación; menor es mejor');ax.legend();fig_h2(18,'comparacion_validacion')
print('* ARIMA sin convergencia en todas las estaciones: se muestra su error, pero no es elegible para selección.')
RESUMEN_H2['seleccion']={'modelo':MODELO_ELEGIDO,'mae_macro_val':float(best.MAE_macro_estacion),
                        'mae_global_val':float(best.MAE_global),'arima':ARIMA_ELEGIDO,
                        'modelos_completados':len(comparacion_val_h2),'errores':errores_modelo,
                        'features':features_modelo.get(MODELO_ELEGIDO,['historia univariada PM2.5'])}


### 4.3. Selección cerrada, prueba final y explicación
Se reentrena el modelo elegido con entrenamiento + validación.

In [ ]:
from sklearn.inspection import permutation_importance
import joblib

pred_test={};modelos_finales={};test_rows=[]
for name,pred in [('Persistencia',test.PM2_5.to_numpy()),
                 ('Estacional24h',test.pm25_estacional_24h.fillna(test.PM2_5).to_numpy())]:
    pred_test[name]=pred

referencias=list(dict.fromkeys([MODELO_ELEGIDO,'Lineal']+([ARIMA_ELEGIDO] if ARIMA_ELEGIDO else [])))
for name in referencias:
    print('PRUEBA_FINAL',name,'reentrenamiento',len(dev),flush=True)
    if name in ARIMA_ORDENES:
        p,ok=predecir_arima(ARIMA_ORDENES[name],test,CORTE_TEST-pd.Timedelta(hours=1),'prueba')
        if not ok: print('ARIMA: se informa advertencia de convergencia en refit final.')
    else:
        feats=features_modelo[name]
        pipe=clone(modelos_entrenados[name])
        pipe.fit(dev[feats],dev[OBJETIVO])
        p=predecir_en_lotes(pipe,test[feats]);modelos_finales[name]=pipe
        joblib.dump({'pipeline':pipe,'features':feats,'horizonte_horas':1,
                     'zona_horaria':'America/Lima','fin_etiquetas_entrenamiento':str(dev.FECHA_OBJETIVO.max())},
                    SALIDA/f'modelo_final_{name}.joblib',compress=3)
    pred_test[name]=np.maximum(p,0)
for name,pred in pred_test.items():
    test_rows.append({'Modelo':name,**evaluar(test,pred)})
resultados_test_h2=tabla_h2(pd.DataFrame(test_rows).rename(columns={'N_validacion':'N_prueba'}),26,'Desempeño inicial en la prueba temporal reservada')
predicciones_test=test[CLAVE+['FECHA_OBJETIVO',OBJETIVO,'PM2_5']].copy()
for name,pred in pred_test.items():predicciones_test['pred_'+name]=pred
predicciones_test.to_csv(SALIDA/'predicciones_prueba.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(registros_arima).to_csv(SALIDA/'auditoria_arima_completa.csv',index=False,encoding='utf-8-sig')

prueba_modelo=pred_test[MODELO_ELEGIDO]
por_estacion=[]
for station,rows in test.groupby('ESTACION'):
    mask=test.ESTACION.eq(station).to_numpy()
    a=mean_absolute_error(rows[OBJETIVO],prueba_modelo[mask])
    b=mean_absolute_error(rows[OBJETIVO],pred_test['Persistencia'][mask])
    por_estacion.append({'ESTACION':station,'Horas evaluadas':len(rows),'MAE modelo':a,'MAE persistencia':b,
                        'RMSE modelo':np.sqrt(mean_squared_error(rows[OBJETIVO],prueba_modelo[mask])),
                        'Mejora MAE %':100*(1-a/b),
                        'Última hora objetivo':str(rows.FECHA_OBJETIVO.max())})
errores_estacion_h2=tabla_h2(por_estacion,27,'Error y mejora respecto de persistencia por estación en prueba')
fig,ax=plt.subplots(figsize=(12,5))
errores_estacion_h2.set_index('ESTACION')[['MAE modelo','MAE persistencia']].plot.barh(ax=ax,color=['#457ea2','#c69b5a'])
ax.set(xlabel='Error absoluto medio (escala del archivo)',title='Desempeño por estación en prueba');fig_h2(21,'errores_estacion')
sample_idx=np.random.default_rng(SEED).choice(len(test),size=min(5000,len(test)),replace=False)
fig,axes=plt.subplots(1,2,figsize=(11,4.5))
truth=test[OBJETIVO].to_numpy()
axes[0].scatter(truth[sample_idx],prueba_modelo[sample_idx],s=6,alpha=.3,color='#447f9f')
maximo=max(truth[sample_idx].max(),prueba_modelo[sample_idx].max());axes[0].plot([0,maximo],[0,maximo],'k--',lw=1)
axes[0].set(xlabel='PM2.5 observado',ylabel='PM2.5 estimado',title=MODELO_ELEGIDO+' — muestra para visualizar')
axes[1].hist(truth-prueba_modelo,bins=80,color='#789e9e',alpha=.85)
axes[1].axvline(0,color='black',lw=1);axes[1].set(xlabel='Observado − estimado',ylabel='Horas',title='Errores en todas las horas de prueba')
fig.suptitle('Correspondencia y errores fuera del entrenamiento');fig_h2(20,'observado_estimado')
p95_entrenamiento=float(train[OBJETIVO].quantile(.95))
analisis_altos=[]
for label,mask in [('Hasta P95 del train',truth<=p95_entrenamiento),('Por encima de P95 del train',truth>p95_entrenamiento)]:
    if mask.sum():analisis_altos.append({'Rango':label,'N':int(mask.sum()),'MAE':mean_absolute_error(truth[mask],prueba_modelo[mask]),
                                       'RMSE':np.sqrt(mean_squared_error(truth[mask],prueba_modelo[mask]))})
display(pd.DataFrame(analisis_altos));pd.DataFrame(analisis_altos).to_csv(SALIDA/'errores_concentraciones_altas.csv',index=False)

# Explicación sobre validación
nombre_interpretado=MODELO_ELEGIDO if MODELO_ELEGIDO in modelos_entrenados else MODELO_TABULAR_ELEGIDO
features_interpretacion=features_modelo[nombre_interpretado]
sample_val=pd.concat([d.sample(min(250,len(d)),random_state=SEED) for _,d in val.groupby('ESTACION')]).sort_index()
def score_macro(estimator,X,y):
    p=np.maximum(estimator.predict(X),0)
    return -pd.DataFrame({'s':X.ESTACION.to_numpy(),'e':np.abs(np.asarray(y)-p)}).groupby('s').e.mean().mean()
print('IMPORTANCIA_EN_VALIDACION',nombre_interpretado,len(sample_val),flush=True)
perm=permutation_importance(modelos_entrenados[nombre_interpretado],sample_val[features_interpretacion],sample_val[OBJETIVO],
                            scoring=score_macro,n_repeats=2,random_state=SEED,n_jobs=1)
importancia_h2=tabla_h2(pd.DataFrame({'Variable':features_interpretacion,
                                   'Aumento MAE macro al permutar':perm.importances_mean,
                                   'Desviación entre repeticiones':perm.importances_std})
                     .sort_values('Aumento MAE macro al permutar',ascending=False).head(15),28,'Dependencia predictiva por permutación en validación (no causalidad)')
v=importancia_h2.sort_values('Aumento MAE macro al permutar')
plt.figure(figsize=(10,6));plt.barh(v.Variable,v['Aumento MAE macro al permutar'],color='#597f9f')
plt.xlabel('Incremento del MAE medio entre estaciones');plt.title('Importancia por permutación — '+nombre_interpretado);fig_h2(22,'importancia')

elegido_test=resultados_test_h2.loc[resultados_test_h2.Modelo.eq(MODELO_ELEGIDO)].iloc[0]
persist_test=resultados_test_h2.loc[resultados_test_h2.Modelo.eq('Persistencia')].iloc[0]
mejora_global=100*(1-elegido_test.MAE_global/persist_test.MAE_global)
mejora_macro=100*(1-elegido_test.MAE_macro_estacion/persist_test.MAE_macro_estacion)
RESUMEN_H2['prueba']={'modelo':MODELO_ELEGIDO,'mae_global':float(elegido_test.MAE_global),
    'mae_macro':float(elegido_test.MAE_macro_estacion),'rmse':float(elegido_test.RMSE),'r2':float(elegido_test.R2),
    'mae_persistencia':float(persist_test.MAE_global),'mae_macro_persistencia':float(persist_test.MAE_macro_estacion),
    'mejora_global_pct':float(mejora_global),'mejora_macro_pct':float(mejora_macro),
    'estaciones_con_mejora':int(errores_estacion_h2['Mejora MAE %'].gt(0).sum()),
    'inicio':str(test.FECHA_OBJETIVO.min()),'fin':str(test.FECHA_OBJETIVO.max()),
    'p95_train':p95_entrenamiento,'errores_altos':analisis_altos}
(SALIDA/'resumen_hito2.json').write_text(json.dumps(RESUMEN_H2,ensure_ascii=False,indent=2),encoding='utf-8')
pd.DataFrame(errores_modelo).to_csv(SALIDA/'modelos_no_completados.csv',index=False)
print('Criterio global frente a persistencia:', 'CUMPLIDO' if mejora_global>0 else 'NO CUMPLIDO')
print('Mejora MAE global:',round(mejora_global,2),'%; mejora MAE macro:',round(mejora_macro,2),'%')


### 4.4. Dashboard retrospectivo
La demostración permite elegir estación y comparar observado, pronóstico a una hora y persistencia en el periodo de prueba.

In [ ]:
import plotly.graph_objects as go

estacion_demo=errores_estacion_h2.sort_values('Horas evaluadas',ascending=False).ESTACION.iloc[0]
def ventana_dashboard(station):
    p=predicciones_test.loc[predicciones_test.ESTACION.eq(station)].set_index('FECHA_OBJETIVO').sort_index()
    end=p.index.max();index=pd.date_range(end-pd.Timedelta(days=7)+pd.Timedelta(hours=1),end,freq='h')
    raw=series_pm[station].reindex(index)
    return pd.DataFrame({'observado':raw,'modelo':p['pred_'+MODELO_ELEGIDO].reindex(index),
                         'persistencia':p.pred_Persistencia.reindex(index)}),end

week,end=ventana_dashboard(estacion_demo)
fig,axes=plt.subplots(2,2,figsize=(13,7),gridspec_kw={'height_ratios':[.7,2]})
axes[0,0].axis('off');axes[0,1].axis('off')
axes[0,0].text(0,.9,'PM2.5 — pronóstico a una hora',fontsize=16,weight='bold')
axes[0,0].text(0,.55,'Modelo: '+MODELO_ELEGIDO+'\nPeriodo de prueba: '+RESUMEN_H2['prueba']['inicio']+' a '+RESUMEN_H2['prueba']['fin'],fontsize=10)
axes[0,1].text(0,.9,f'MAE global: {elegido_test.MAE_global:.3f}    RMSE: {elegido_test.RMSE:.3f}',fontsize=14)
axes[0,1].text(0,.52,f'Mejora MAE frente a persistencia: {mejora_global:.2f} %\nEstaciones con mejora: {RESUMEN_H2["prueba"]["estaciones_con_mejora"]}/7',fontsize=11)
axes[1,0].plot(week.index,week.observado,label='Observado',color='#395a7a',lw=1.3)
axes[1,0].plot(week.index,week.modelo,label='Pronóstico t+1 h',color='#cc7252',lw=1.1)
axes[1,0].plot(week.index,week.persistencia,label='Persistencia',color='#999999',alpha=.6,lw=.8)
axes[1,0].set(title=estacion_demo.replace('_',' ')+' — últimos siete días evaluables',xlabel='Fecha objetivo, hora Perú',ylabel='PM2.5 (µg/m³)');axes[1,0].legend(fontsize=8)
axes[1,0].tick_params(axis='x',rotation=25)
e=errores_estacion_h2.sort_values('MAE modelo')
axes[1,1].barh(e.ESTACION.str.replace('_',' '),e['MAE modelo'],color='#6287a5');axes[1,1].set(title='Error por estación en toda la prueba',xlabel='MAE');axes[1,1].tick_params(axis='y',labelsize=8)
fig.suptitle('Dashboard preliminar: resultados retrospectivos de prueba',y=1.02);fig_h2(23,'dashboard_preliminar')

dashboard=go.Figure()
all_stations=sorted(test.ESTACION.unique())
for i,station in enumerate(all_stations):
    w,limit=ventana_dashboard(station)
    for label,col,color in [('Observado','observado','#345c82'),('Modelo t+1 h','modelo','#d87754'),('Persistencia','persistencia','#888888')]:
        dashboard.add_trace(go.Scatter(x=w.index,y=w[col],mode='lines',name=label,
            line={'color':color,'width':1.7},visible=i==0,connectgaps=False))
buttons=[]
for i,station in enumerate(all_stations):
    mask=[False]*len(dashboard.data)
    for k in range(3):mask[i*3+k]=True
    buttons.append({'label':station.replace('_',' '),'method':'update',
        'args':[{'visible':mask},{'title':f'{station.replace("_"," ")} | {MODELO_ELEGIDO} | Horizonte 1 h | Prueba retrospectiva'}]})
dashboard.update_layout(title=f'{all_stations[0].replace("_"," ")} | {MODELO_ELEGIDO} | Horizonte 1 h | Prueba retrospectiva',
    height=500,xaxis_title='Fecha objetivo — hora oficial del Perú',yaxis_title='PM2.5 (µg/m³)',
    template='plotly_white',updatemenus=[{'buttons':buttons,'x':0,'y':1.18,'xanchor':'left'}],
    legend={'orientation':'h','y':-.2},margin={'t':110,'b':90})
dashboard.update_xaxes(rangeslider_visible=True)
display(HTML(dashboard.to_html(full_html=False,include_plotlyjs=True)))

ar_result=comparacion_val_h2.loc[comparacion_val_h2.Modelo.eq(ARIMA_ELEGIDO)].iloc[0] if ARIMA_ELEGIDO else None
ab_inicio=ablacion_h2.iloc[0];ab_final=ablacion_h2.iloc[-1]
ab_historia=ablacion_h2.loc[ablacion_h2.Configuración.eq('HGB_historia')].iloc[0]
efecto_historia=100*(1-ab_historia.MAE_macro_estacion/ab_inicio.MAE_macro_estacion)
efecto_complementos=100*(1-ab_final.MAE_macro_estacion/ab_historia.MAE_macro_estacion)
veredicto='redujo' if mejora_global>0 else 'no redujo'
ar_texto=(f'El mejor ARIMA fue {ARIMA_ELEGIDO}, con MAE medio entre estaciones de {ar_result.MAE_macro_estacion:.3f} en validación.'
          if ARIMA_ELEGIDO else 'Ninguna configuración ARIMA completó todos los ajustes con convergencia; se registra esta limitación en la Tabla 24.')
